# Step 0: Importing Api and Packages

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
sns.set('notebook', font_scale=1.25, style='whitegrid')

from pathlib import Path
import re

import nba_api
import nba_api.stats.endpoints as ep
from nba_api.stats.static import teams

import xgboost as xgb

In [2]:
from nba_api.stats.endpoints import (
    leaguedashplayerstats,
    PlayerEstimatedMetrics,
    ScheduleLeagueV2,
    LeagueDashPlayerBioStats,
    GravityLeaders,
    LeagueDashPlayerShotLocations,
    LeagueDashPlayerStats,
    LeagueDashPtDefend,
    LeagueDashPlayerPtShot,
    ScheduleLeagueV2,
)

# Seasons to pull
SEASONS = [
    "2021-22",
    "2022-23",
    "2023-24",
    "2024-25",
    "2025-26",
    "2026-27",
]

endpoints = {
    "ScheduleLeagueV2": ScheduleLeagueV2,
    "LeagueDashPlayerBioStats": LeagueDashPlayerBioStats,
    "GravityLeaders": GravityLeaders,
    "LeagueDashPlayerShotLocations": LeagueDashPlayerShotLocations,
    "LeagueDashPlayerStats": LeagueDashPlayerStats,
    "LeagueDashPtDefend": LeagueDashPtDefend,
    "LeagueDashPlayerPtShot": LeagueDashPlayerPtShot,
    "ScheduleLeagueV2": ScheduleLeagueV2,
}

# Helper Function Used 

## Retrieving Data 

# Step 1: Retrieve Dataframes From NBA API

To improve performance, I’ll pull the remote API dataframes locally so they can be accessed more efficiently during manipulation and analysis.

In [ ]:
OUTPUT_DIR = "/Users/ronanhwang/Downloads/Github/nba_2020s/csv_files"

os.makedirs(OUTPUT_DIR, exist_ok = True)

# Helper function to convert SEASONS to seasons in api
def season_filename(season, endpoint_name):
    start, end = season.split("-")
    season_short = f"{start[-2:]}_{end[-2:]}"
    return f"{season_short}{endpoint_name}.csv"


for season in SEASONS:

    print(f"SEASON: {season}")

    for name, endpoint in endpoints.items():

        filename = season_filename(season, name)

        filepath = os.path.join(OUTPUT_DIR, filename)

        print(f"\nPulling {name}...")

        try:
            # Different endpoints have different args
            if name == "ScheduleLeagueV2":

                response = endpoint(
                    league_id="00",
                    season=season,
                )

            else:

                response = endpoint(
                    season=season,
                    season_type_all_star="Regular Season",
                )

            # Get DataFrame
            df = response.get_data_frames()[0]

            # Export
            df.to_csv(filepath, index = False)

            print(f"Exported: {filename}")
            print(f"Shape: {df.shape}")

            # Read CSV back in
            df_check = pd.read_csv(filepath)

            print(f"\nHead of {filename}:")
            print(df_check.head())

            # Look for season columns
            season_cols = [
                col
                for col in df_check.columns
                if "SEASON" in col.upper()
                or "YEAR" in col.upper()
            ]

            if season_cols:
                print("\nSeason information:")

                for col in season_cols:
                    print(
                        f"  {col}: "
                        f"{df_check[col].dropna().unique()[:10]}"
                    )

        except Exception as e:

            print(f"ERROR pulling {name} for {season}:")
            print(e)


# Step 2: Read in Data 

In [3]:
CSV_DIR = Path("/Users/ronanhwang/Downloads/Github/nba_2020s/csv_files")


# Helper function that laods all NBA CSV files from csv_files directory
# Expects 23_24LeagueDashPlayerStats.csv format
# Returns 
    # data = {
        # "23_24": {
        #     "LeagueDashPlayerStats": DataFrame,
        #     "GravityLeaders": DataFrame,
        #     ...
        # },
def load_nba_csvs(csv_dir):
    data = {}

    # Loop through every CSV in the directory
    for filepath in csv_dir.glob("*.csv"):

        filename = filepath.name

        # Extract season from beginning of filename
        match = re.match(r"^(\d{2}_\d{2})(.+)\.csv$", filename)

        if not match:
            print(f"Skipping unrecognized filename: {filename}")
            continue

        season = match.group(1)
        endpoint = match.group(2)

        # Create season dictionary if necessary
        if season not in data:
            data[season] = {}

        # Read CSV
        df = pd.read_csv(filepath)

        # Store DataFrame
        data[season][endpoint] = df

        print(f"{filename} -> {season} / {endpoint} | Shape: {df.shape}")

    return data


In [4]:
data = load_nba_csvs(CSV_DIR)

24_25ScheduleLeagueV2.csv -> 24_25 / ScheduleLeagueV2 | Shape: (1400, 159)
22_23LeagueDashPtDefend.csv -> 22_23 / LeagueDashPtDefend | Shape: (535, 14)
21_22LeagueDashPlayerStats.csv -> 21_22 / LeagueDashPlayerStats | Shape: (605, 67)
23_24LeagueDashPlayerShotLocations.csv -> 23_24 / LeagueDashPlayerShotLocations | Shape: (573, 30)
23_24LeagueDashPlayerPtShot.csv -> 23_24 / LeagueDashPlayerPtShot | Shape: (568, 20)
23_24ScheduleLeagueV2.csv -> 23_24 / ScheduleLeagueV2 | Shape: (1396, 184)
21_22LeagueDashPlayerPtShot.csv -> 21_22 / LeagueDashPlayerPtShot | Shape: (596, 20)
25_26Misc.csv -> 25_26 / Misc | Shape: (582, 20)
23_24LeagueDashPlayerBioStats.csv -> 23_24 / LeagueDashPlayerBioStats | Shape: (572, 23)
25_26LeagueDashPlayerPtShot.csv -> 25_26 / LeagueDashPlayerPtShot | Shape: (582, 20)
21_22LeagueDashPtDefend.csv -> 21_22 / LeagueDashPtDefend | Shape: (595, 14)
24_25LeagueDashPlayerBioStats.csv -> 24_25 / LeagueDashPlayerBioStats | Shape: (569, 23)
25_26LeagueDashPlayerStats.csv -

## See all .csv files 

In [5]:
for season, endpoints in data.items():
    print(f"\n{season}")

    for endpoint, df in endpoints.items():
        print(f"  {endpoint}: {df.shape}")


24_25
  ScheduleLeagueV2: (1400, 159)
  LeagueDashPlayerBioStats: (569, 23)
  LeagueDashPlayerShotLocations: (570, 30)
  LeagueDashPlayerPtShot: (566, 20)
  LeagueDashPlayerStats: (569, 67)
  LeagueDashPtDefend: (568, 14)
  Misc: (569, 20)

22_23
  LeagueDashPtDefend: (535, 14)
  ScheduleLeagueV2: (1394, 151)
  LeagueDashPlayerStats: (539, 67)
  Misc: (539, 20)
  LeagueDashPlayerPtShot: (537, 20)
  LeagueDashPlayerBioStats: (539, 23)
  LeagueDashPlayerShotLocations: (540, 30)

21_22
  LeagueDashPlayerStats: (605, 67)
  LeagueDashPlayerPtShot: (596, 20)
  LeagueDashPtDefend: (595, 14)
  LeagueDashPlayerBioStats: (605, 23)
  ScheduleLeagueV2: (1393, 152)
  LeagueDashPlayerShotLocations: (606, 30)
  Misc: (605, 20)

23_24
  LeagueDashPlayerShotLocations: (573, 30)
  LeagueDashPlayerPtShot: (568, 20)
  ScheduleLeagueV2: (1396, 184)
  LeagueDashPlayerBioStats: (572, 23)
  LeagueDashPlayerStats: (572, 67)
  LeagueDashPtDefend: (569, 14)
  Misc: (572, 20)

25_26
  Misc: (582, 20)
  LeagueDas

# Step 3: Cleaning Data

In [6]:
# Helper function that cleans all NBA DataFrames stored in the data dictionary by dropping/renaming
# Expects: data[season][endpoint] = DataFrame
# Returns: Cleaned data dictionary.
def clean_nba_data(data):
    
    # Columns to drop from LeagueDashPlayerStats
    player_stats_drop = [
        'WNBA_FANTASY_PTS', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK',
        'MIN_RANK', 'FGM_RANK', 'FGA_RANK', 'FG_PCT_RANK', 'FG3M_RANK',
        'FG3A_RANK', 'FG3_PCT_RANK', 'FTM_RANK', 'FTA_RANK', 'FT_PCT_RANK',
        'OREB_RANK', 'DREB_RANK', 'REB_RANK', 'AST_RANK', 'TOV_RANK',
        'STL_RANK', 'BLK_RANK', 'BLKA_RANK', 'PF_RANK', 'PFD_RANK',
        'PTS_RANK', 'PLUS_MINUS_RANK', 'NBA_FANTASY_PTS_RANK', 'DD2_RANK',
        'TD3_RANK', 'WNBA_FANTASY_PTS_RANK', 'FP_HIGH_SCORE_RANK',
        'TEAM_COUNT'
    ]

    # Columns to drop from LeagueDashPtDefend
    pt_defend_drop = [
        'AGE', 'GP', 'G'
    ]

    # Rename LeagueDashPtDefend columns
    pt_defend_rename = {
        "CLOSE_DEF_PERSON_ID": "D_PLAYER_ID",
        "PLAYER_NAME": "D_PLAYER_NAME",
        "PLAYER_LAST_TEAM_ID": "D_PLAYER_LAST_TEAM_ID",
        "PLAYER_LAST_TEAM_ABBREVIATION": "D_PLAYER_LAST_TEAM_ABBREVIATION",
        "PLAYER_POSITION": "D_PLAYER_POSITION",
        "FREQ": "D_FREQ",
        "NORMAL_FG_PCT": "D_NORMAL_FG_PCT",
        "PCT_PLUSMINUS": "D_PCT_PLUSMINUS",
    }

    # Rename LeagueDashPlayerShotLocations columns
    shot_locations_rename = {
        "Unnamed: 0": "PLAYER_ID",
        "Unnamed: 1": "PLAYER_NAME",
        "Unnamed: 2": "TEAM_ID",
        "Unnamed: 3": "TEAM_ABBREVIATION",
        "Unnamed: 4": "AGE",
        "Unnamed: 5": "NICKNAME",
    }

    # Columns to drop from LeagueDashPlayerPtShot
    pt_shot_drop = [
        'AGE', 'GP', 'FGM', 'FGA',
        'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT'
    ]

    # Columns to keep from ScheduleLeagueV2
    schedule_keep = [
        'seasonYear',
        'gameDate',
        'gameId',
        'gameCode',
        'gameStatusText',
        'gameSequence',
        'gameDateTimeEst',
        'seriesGameNumber',
        'gameLabel',
        'gameSubLabel',
        'seriesText',
        'postponedStatus',
        'isNeutral',
        'homeTeam_teamId',
        'homeTeam_teamTricode',
        'homeTeam_wins',
        'homeTeam_losses',
        'homeTeam_score',
        'awayTeam_teamId',
        'awayTeam_teamTricode',
        'awayTeam_wins',
        'awayTeam_losses',
        'awayTeam_score'
    ]

    # Columns to keep from Misc
    misc_keep = [
        'Player',
        'TEAM',
        'AGE',
        'PTS\nOFF TO',
        '2nd\nPTS',
        'FBPs',
        'PITP',
        'Opp PTS\nOFF TO',
        'Opp\n2nd PTS',
        'Opp\nFBPs',
        'Opp\nPITP',
        'BLK',
        'BLKA'
    ]


    # Loop through every season
    for season, endpoints in data.items():

        print(f"\nCleaning {season}...")

        # LeagueDashPlayerStats
        if "LeagueDashPlayerStats" in endpoints:
            endpoints["LeagueDashPlayerStats"].drop(
                columns=player_stats_drop,
                errors="ignore",
                inplace=True
            )
        
        # LeagueDashPtDefend
        if "LeagueDashPtDefend" in endpoints:
            endpoints["LeagueDashPtDefend"].drop(
                columns=pt_defend_drop,
                errors="ignore",
                inplace=True
            )

            endpoints["LeagueDashPtDefend"].rename(
                columns=pt_defend_rename,
                inplace=True
            )
        
        # LeagueDashPlayerShotLocations
        if "LeagueDashPlayerShotLocations" in endpoints:
            endpoints["LeagueDashPlayerShotLocations"].rename(
                columns=shot_locations_rename,
                inplace=True
            )

            # Shot-location NaNs represent no recorded shots in that category
            endpoints["LeagueDashPlayerShotLocations"].fillna(0, inplace=True)
        
        # LeagueDashPlayerPtShot
        if "LeagueDashPlayerPtShot" in endpoints:
            endpoints["LeagueDashPlayerPtShot"].drop(
                columns=pt_shot_drop,
                errors="ignore",
                inplace=True
            )

        # ScheduleLeagueV2
        if "ScheduleLeagueV2" in endpoints:
            df = endpoints["ScheduleLeagueV2"]

            endpoints["ScheduleLeagueV2"] = df[
                [
                    col
                    for col in schedule_keep
                    if col in df.columns
                ]
            ]

        # Misc
        if "Misc" in endpoints:
            df = endpoints["Misc"]

            # Keep only desired columns
            endpoints["Misc"] = df[
                [
                    col
                    for col in misc_keep
                    if col in df.columns
                ]
            ]

            # Fill missing stat values with 0
            misc_stat_columns = [
                col for col in misc_keep
                if col not in ["Player", "TEAM"]
                and col in endpoints["Misc"].columns
            ]

            endpoints["Misc"][misc_stat_columns] = (
                endpoints["Misc"][misc_stat_columns].fillna(0)
            )

        print(f"Finished {season}")

    return data


Note that the data is cleaned within the notebook and is not updated in the original files. Therefore, the folders in the directory still contain the original, uncleaned data.

In [7]:
data = clean_nba_data(data)


Cleaning 24_25...
Finished 24_25

Cleaning 22_23...
Finished 22_23

Cleaning 21_22...
Finished 21_22

Cleaning 23_24...
Finished 23_24

Cleaning 25_26...
Finished 25_26


/var/folders/0b/n3f548tn3156qpndhm9hlkhh0000gn/T/ipykernel_22896/2089117680.py:172: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  endpoints["Misc"][misc_stat_columns] = (
/var/folders/0b/n3f548tn3156qpndhm9hlkhh0000gn/T/ipykernel_22896/2089117680.py:172: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  endpoints["Misc"][misc_stat_columns] = (
/var/folders/0b/n3f548tn3156qpndhm9hlkhh0000gn/T/ipykernel_22896/2089117680.py:172: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataF

# Step 4: Merging Dataframes Together

In [8]:
# Helper function that merges Stats, Shot Locations, Defense,
# Player Shot data, and Misc data for every season.
# Expected structure: data[season][endpoint] = DataFrame
# Returns: merged_data[season] = merged DataFrame
def merge_nba_data(data):

    merged_data = {}

    shot_columns = [
        "Restricted Area", "Restricted Area.1", "Restricted Area.2",
        "In The Paint (Non-RA)", "In The Paint (Non-RA).1", "In The Paint (Non-RA).2",
        "Mid-Range", "Mid-Range.1", "Mid-Range.2",
        "Left Corner 3", "Left Corner 3.1", "Left Corner 3.2",
        "Right Corner 3", "Right Corner 3.1", "Right Corner 3.2",
        "Above the Break 3", "Above the Break 3.1", "Above the Break 3.2",
        "Backcourt", "Backcourt.1", "Backcourt.2",
        "Corner 3", "Corner 3.1", "Corner 3.2"
    ]

    # Normalize player names ONLY for matching across datasets
    def normalize_player_name(name):

        if pd.isna(name):
            return name

        name = str(name).strip()

        # Normalize Jr. -> Jr
        name = name.replace(" Jr.", " Jr")

        # Known cross-source naming difference
        name_overrides = {
            "Bobby Portis Jr": "Bobby Portis",
            "Brandon Boston Jr": "Brandon Boston",
            "Jonas Valanciunas": 'Jonas Valančiūnas',
            "Lester Quiñones": 'Lester Quinones',
            "Vlatko Cancar": 'Vlatko Čančar',
        }


        return name_overrides.get(name, name)

    for season, endpoints in data.items():

        print(f"\nMerging {season}...")

        # Get DataFrames
        stats = endpoints.get("LeagueDashPlayerStats")
        shots = endpoints.get("LeagueDashPlayerShotLocations")
        defense = endpoints.get("LeagueDashPtDefend")
        freq = endpoints.get("LeagueDashPlayerPtShot")
        misc = endpoints.get("Misc")

        # Check required DataFrames
        if stats is None:
            print(f"  Skipping {season}: missing Player Stats")
            continue

        if shots is None:
            print(f"  Skipping {season}: missing Shot Locations")
            continue

        if defense is None:
            print(f"  Skipping {season}: missing Defense")
            continue

        if freq is None:
            print(f"  Skipping {season}: missing Player Shot data")
            continue

        if misc is None:
            print(f"  Skipping {season}: missing Misc data")
            continue

        # Make PLAYER_ID types consistent
        stats["PLAYER_ID"] = pd.to_numeric(
            stats["PLAYER_ID"],
            errors="coerce"
        ).astype("Int64")

        shots["PLAYER_ID"] = pd.to_numeric(
            shots["PLAYER_ID"],
            errors="coerce"
        ).astype("Int64")

        # Defense has ALREADY been renamed to D_PLAYER_ID
        defense["D_PLAYER_ID"] = pd.to_numeric(
            defense["D_PLAYER_ID"],
            errors="coerce"
        ).astype("Int64")

        freq["PLAYER_ID"] = pd.to_numeric(
            freq["PLAYER_ID"],
            errors="coerce"
        ).astype("Int64")

        # Clean player names
        stats["PLAYER_NAME"] = (
            stats["PLAYER_NAME"]
            .astype(str)
            .str.strip()
        )

        misc["Player"] = (
            misc["Player"]
            .astype(str)
            .str.strip()
        )

        # 1. Stats + Shot Locations
        merged = stats.merge(
            shots,
            on="PLAYER_ID",
            how="left",
            suffixes=("", "_SHOT")
        )

        # Fill NaNs CREATED by the LEFT JOIN in shot-location columns only
        existing_shot_columns = [
            col for col in shot_columns
            if col in merged.columns
        ]

        merged[existing_shot_columns] = (
            merged[existing_shot_columns].fillna(0)
        )

        # 2. Merge Defense
        merged = merged.merge(
            defense,
            left_on="PLAYER_ID",
            right_on="D_PLAYER_ID",
            how="left"
        )

        # 3. Merge Player Shot/Frequency Data
        merged = merged.merge(
            freq,
            on="PLAYER_ID",
            how="left",
            suffixes=("", "_CLUTCH")
        )

        # 4. Normalize names for Misc merge
        merged["_MISC_PLAYER_NAME"] = (
            merged["PLAYER_NAME"]
            .apply(normalize_player_name)
        )

        misc["_MISC_PLAYER_NAME"] = (
            misc["Player"]
            .apply(normalize_player_name)
        )

        # 5. Merge Misc
        merged = merged.merge(
            misc,
            on="_MISC_PLAYER_NAME",
            how="left",
            suffixes=("", "_MISC")
        )

        # Remove temporary merge key
        merged.drop(
            columns="_MISC_PLAYER_NAME",
            inplace=True
        )

        # Store merged DataFrame
        merged_data[season] = merged

        print(f"  Merged shape: {merged.shape}")

    return merged_data


In [9]:
merged_data = merge_nba_data(data)


Merging 24_25...
  Merged shape: (569, 99)

Merging 22_23...
  Merged shape: (540, 99)

Merging 21_22...
  Merged shape: (605, 99)

Merging 23_24...
  Merged shape: (572, 99)

Merging 25_26...
  Merged shape: (582, 100)


# Step 5: Check Missing Data

In [10]:
# Helper function that inspects rows containing NaN values across all merged seasons
# Expected structure: merged_data[season] = DataFrame
# Returns: None
def inspect_missing_rows(merged_data):

    for season, df in merged_data.items():

        print(f"SEASON: {season}")

        # Find columns containing NaN
        missing_counts = df.isna().sum()
        missing_columns = missing_counts[missing_counts > 0]

        if missing_columns.empty:
            print("\nNo missing values.")
            continue

        # Print rows containing missing values
        rows_with_nan = df[df.isna().any(axis=1)]

        print(f"ROWS WITH MISSING VALUES: {len(rows_with_nan):,}")

        print(rows_with_nan.to_string(index=False))

In [11]:
inspect_missing_rows(merged_data)

SEASON: 24_25
ROWS WITH MISSING VALUES: 9
 PLAYER_ID     PLAYER_NAME NICKNAME    TEAM_ID TEAM_ABBREVIATION  AGE  GP  W  L  W_PCT       MIN  FGM  FGA  FG_PCT  FG3M  FG3A  FG3_PCT  FTM  FTA  FT_PCT  OREB  DREB  REB  AST  TOV  STL  BLK  BLKA  PF  PFD  PTS  PLUS_MINUS  NBA_FANTASY_PTS  DD2  TD3 PLAYER_NAME_SHOT TEAM_ID_SHOT TEAM_ABBREVIATION_SHOT AGE_SHOT NICKNAME_SHOT Restricted Area Restricted Area.1 Restricted Area.2 In The Paint (Non-RA) In The Paint (Non-RA).1 In The Paint (Non-RA).2 Mid-Range Mid-Range.1 Mid-Range.2 Left Corner 3 Left Corner 3.1 Left Corner 3.2 Right Corner 3 Right Corner 3.1 Right Corner 3.2 Above the Break 3 Above the Break 3.1 Above the Break 3.2 Backcourt Backcourt.1 Backcourt.2 Corner 3 Corner 3.1 Corner 3.2  D_PLAYER_ID   D_PLAYER_NAME  D_PLAYER_LAST_TEAM_ID D_PLAYER_LAST_TEAM_ABBREVIATION D_PLAYER_POSITION  D_FREQ  D_FGM  D_FGA  D_FG_PCT  D_NORMAL_FG_PCT  D_PCT_PLUSMINUS PLAYER_NAME_CLUTCH  PLAYER_LAST_TEAM_ID PLAYER_LAST_TEAM_ABBREVIATION   G  FGA_FREQUENCY  

# Step 6: Create Dataframe with Winning features

In [12]:
col_Altogether = [

    # Player stats
    'PLAYER_ID',
    'PLAYER_NAME',
    'TEAM_ID',
    'TEAM_ABBREVIATION',

    'AGE',
    'GP',
    'W',
    'L',
    'W_PCT',
    'MIN',

    'FGM',
    'FGA',
    'FG_PCT',

    'FG3M',
    'FG3A',
    'FG3_PCT',

    'FTM',
    'FTA',
    'FT_PCT',

    'OREB',
    'DREB',
    'REB',
    'AST',
    'TOV',
    'STL',
    'BLK',
    'BLKA',
    'PF',
    'PFD',

    'PTS',
    'PLUS_MINUS',
    'NBA_FANTASY_PTS',
    'DD2',
    'TD3',

    # Shot locations
    # Base   = FGM
    # .1     = FGA
    # .2     = FG%
    'Restricted Area',
    'Restricted Area.1',
    'Restricted Area.2',

    'In The Paint (Non-RA)',
    'In The Paint (Non-RA).1',
    'In The Paint (Non-RA).2',

    'Mid-Range',
    'Mid-Range.1',
    'Mid-Range.2',

    'Left Corner 3',
    'Left Corner 3.1',
    'Left Corner 3.2',

    'Right Corner 3',
    'Right Corner 3.1',
    'Right Corner 3.2',

    'Above the Break 3',
    'Above the Break 3.1',
    'Above the Break 3.2',

    'Backcourt',
    'Backcourt.1',
    'Backcourt.2',

    # Aggregate of Left Corner 3 + Right Corner 3
    'Corner 3',
    'Corner 3.1',
    'Corner 3.2',

    # Defense
    'D_PLAYER_POSITION',
    'D_FREQ',
    'D_FGM',
    'D_FGA',
    'D_FG_PCT',
    'D_NORMAL_FG_PCT',
    'D_PCT_PLUSMINUS',

    # Player shot
    'FGA_FREQUENCY',
    'EFG_PCT',
    'FG2A_FREQUENCY',
    'FG2M',
    'FG2A',
    'FG2_PCT',
    'FG3A_FREQUENCY',

    # Misc
    'PTSOFFTO',
    '2ndPTS',
    'FBPs',
    'PITP',

    'Opp PTSOFF TO',
    'Opp2nd PTS',
    'OppFBPs',
    'OppPITP'
]


In [13]:
# Helper function that creates the final player-level dataset for every season.
# Expected structure: merged_data[season] = DataFrame
# Returns: player_data[season] = transformed DataFrame
def create_player_dataset(merged_data):

    player_data = {}

    for season, df in merged_data.items():

        print(f"\nCreating player dataset for {season}...")

        # Rename miscellaneous source columns
        df = df.rename(columns={
            'PTS\nOFF TO': 'PTSOFFTO',
            '2nd\nPTS': '2ndPTS',
            'Opp PTS\nOFF TO': 'Opp PTSOFF TO',
            'Opp\n2nd PTS': 'Opp2nd PTS',
            'Opp\nFBPs': 'OppFBPs',
            'Opp\nPITP': 'OppPITP'
        })

        # Start with the columns from col_Altogether
        output = df[
            [
                col
                for col in col_Altogether
                if col in df.columns
            ]
        ].copy()

        # Rename columns 
        output = output.rename(columns={
            'PLAYER_ID': 'PlayerID',
            'PLAYER_NAME': 'PlayerName',
            'TEAM_ID': 'TeamID',
            'TEAM_ABBREVIATION': 'TeamAbbv',

            'AGE': 'Age',
            'W_PCT': 'W_Pct',
            'MIN': 'Min',

            'FG3M': '3PM',
            'OREB': 'OReb',
            'DREB': 'DReb',
            'AST': 'Ast',
            'TOV': 'TO',
            'PTS': 'Pts',

            'D_PLAYER_POSITION': 'Pos'
        })

        # Remaining games
        output['RemainingGames'] = output['GP']

        # Paint points
        paint_fgm = (
            output['Restricted Area']
            .fillna(0)
            .astype(float)
            +
            output['In The Paint (Non-RA)']
            .fillna(0)
            .astype(float)
        )

        output['Pts_Paint'] = paint_fgm * 2

        # Mid-range points
        midrange_fgm = (
            output['Mid-Range']
            .fillna(0)
            .astype(float)
        )

        output['Pts_MidRange'] = midrange_fgm * 2

        # 3-point points
        three_fgm = (
            output['Left Corner 3']
            .fillna(0)
            .astype(float)
            +
            output['Right Corner 3']
            .fillna(0)
            .astype(float)
            +
            output['Above the Break 3']
            .fillna(0)
            .astype(float)
            +
            output['Backcourt']
            .fillna(0)
            .astype(float)
        )

        output['Pts_3PT'] = three_fgm * 3

        # Store transformed player-level dataset
        player_data[season] = output

        print(
            f"Finished {season}: "
            f"{output.shape[0]:,} rows × "
            f"{output.shape[1]:,} columns"
        )

    return player_data


In [14]:
player_data = create_player_dataset(merged_data)


Creating player dataset for 24_25...
Finished 24_25: 569 rows × 84 columns

Creating player dataset for 22_23...
Finished 22_23: 540 rows × 84 columns

Creating player dataset for 21_22...
Finished 21_22: 605 rows × 84 columns

Creating player dataset for 23_24...
Finished 23_24: 572 rows × 84 columns

Creating player dataset for 25_26...
Finished 25_26: 582 rows × 84 columns


# Step 7: Create Dataframe with Player Per Game Averages

In [15]:
# Features in the dataframe that I want to shift to PerGame and PerSeason
per_game_cols = [
    'Min', 'FGM', 'FGA', '3PM', 'FG3A',
    'FTM', 'FTA',
    'OReb', 'DReb', 'REB',
    'Ast', 'TO', 'STL', 'BLK', 'BLKA',
    'PF', 'PFD', 'Pts',
    'NBA_FANTASY_PTS', 'DD2', 'TD3',
    'Pts_Paint', 'Pts_MidRange', 'Pts_3PT', 'PLUS_MINUS'
]

feature_cols = [
    'PlayerID', 'PlayerName', 'TeamID', 'TeamAbbv', 'Age', 'GP',
    'W', 'L', 'W_Pct',
    'FG_PCT', 'FG3_PCT', 'FT_PCT',
    'Restricted Area', 'Restricted Area.1', 'Restricted Area.2',
    'In The Paint (Non-RA)', 'In The Paint (Non-RA).1', 'In The Paint (Non-RA).2',
    'Mid-Range', 'Mid-Range.1', 'Mid-Range.2',
    'Left Corner 3', 'Left Corner 3.1', 'Left Corner 3.2',
    'Right Corner 3', 'Right Corner 3.1', 'Right Corner 3.2',
    'Above the Break 3', 'Above the Break 3.1', 'Above the Break 3.2',
    'Backcourt', 'Backcourt.1', 'Backcourt.2',
    'Corner 3', 'Corner 3.1', 'Corner 3.2',
    'Pos',
    'D_FREQ', 'D_FGM', 'D_FGA', 'D_FG_PCT', 'D_NORMAL_FG_PCT',
    'D_PCT_PLUSMINUS',
    'FGA_FREQUENCY', 'EFG_PCT',
    'FG2A_FREQUENCY', 'FG2M', 'FG2A', 'FG2_PCT',
    'FG3A_FREQUENCY',
    'PTSOFFTO', '2ndPTS', 'FBPs', 'PITP',
    'Opp PTSOFF TO', 'Opp2nd PTS', 'OppFBPs', 'OppPITP',
    'RemainingGames'
]

In [16]:
# Helper function that creates a per-game version of each season's player dataset.
#     Counting stats are divided by GP.
#     All other feature columns are carried over unchanged.
# Expected structure: data[season] = DataFrame
# Returns: per_game_data[season] = DataFrame
def create_per_game_avg(data):
    per_game_data = {}

    for season, df in data.items():

        print(f"Creating per-game averages for {season}...")

        # Only keep columns that actually exist
        available_features = [
            col for col in feature_cols
            if col in df.columns
        ]

        available_per_game = [
            col for col in per_game_cols
            if col in df.columns
        ]

        # Start with non-per-game features
        output = df[available_features].copy()

        # Calculate per-game averages
        for col in available_per_game:

            output[col] = (
                pd.to_numeric(df[col], errors='coerce')
                / pd.to_numeric(df['GP'], errors='coerce')
            )

        per_game_data[season] = output

        print(
            f"Finished {season}: "
            f"{output.shape[0]:,} rows × "
            f"{output.shape[1]:,} columns"
        )

    return per_game_data


In [17]:
PerGameAvg = create_per_game_avg(player_data)

Creating per-game averages for 24_25...
Finished 24_25: 569 rows × 84 columns
Creating per-game averages for 22_23...
Finished 22_23: 540 rows × 84 columns
Creating per-game averages for 21_22...
Finished 21_22: 605 rows × 84 columns
Creating per-game averages for 23_24...
Finished 23_24: 572 rows × 84 columns
Creating per-game averages for 25_26...
Finished 25_26: 582 rows × 84 columns


# Step 8: Generate team rosters for each season

In [18]:
# Helper function that creates a dictionary of NBA team rosters from player data.
# Expected structure: data[season] = DataFrame
# Required columns: TeamAbbv
# Returns: rosters[season][team_abbreviation] = DataFrame
def create_team_rosters(data):
    team_rosters = {}

    for season, df in data.items():

        if "TeamAbbv" not in df.columns:
            raise ValueError(
                f"{season} is missing 'TeamAbbv'"
            )

        team_rosters[season] = {}

        teams = sorted(
            df["TeamAbbv"]
            .dropna()
            .unique()
        )

        for team in teams:

            team_rosters[season][team] = df[
                df["TeamAbbv"] == team
            ]

        print(
            f"{season}: "
            f"{len(teams)} teams"
        )

    return team_rosters


In [19]:
team_rosters = create_team_rosters(PerGameAvg)

24_25: 30 teams
22_23: 30 teams
21_22: 30 teams
23_24: 30 teams
25_26: 30 teams


Debugging

In [20]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

# team_rosters["24_25"]["BOS"].iloc[[0]]
team_rosters["24_25"]["BOS"].columns.tolist()

['PlayerID',
 'PlayerName',
 'TeamID',
 'TeamAbbv',
 'Age',
 'GP',
 'W',
 'L',
 'W_Pct',
 'FG_PCT',
 'FG3_PCT',
 'FT_PCT',
 'Restricted Area',
 'Restricted Area.1',
 'Restricted Area.2',
 'In The Paint (Non-RA)',
 'In The Paint (Non-RA).1',
 'In The Paint (Non-RA).2',
 'Mid-Range',
 'Mid-Range.1',
 'Mid-Range.2',
 'Left Corner 3',
 'Left Corner 3.1',
 'Left Corner 3.2',
 'Right Corner 3',
 'Right Corner 3.1',
 'Right Corner 3.2',
 'Above the Break 3',
 'Above the Break 3.1',
 'Above the Break 3.2',
 'Backcourt',
 'Backcourt.1',
 'Backcourt.2',
 'Corner 3',
 'Corner 3.1',
 'Corner 3.2',
 'Pos',
 'D_FREQ',
 'D_FGM',
 'D_FGA',
 'D_FG_PCT',
 'D_NORMAL_FG_PCT',
 'D_PCT_PLUSMINUS',
 'FGA_FREQUENCY',
 'EFG_PCT',
 'FG2A_FREQUENCY',
 'FG2M',
 'FG2A',
 'FG2_PCT',
 'FG3A_FREQUENCY',
 'PTSOFFTO',
 '2ndPTS',
 'FBPs',
 'PITP',
 'Opp PTSOFF TO',
 'Opp2nd PTS',
 'OppFBPs',
 'OppPITP',
 'RemainingGames',
 'Min',
 'FGM',
 'FGA',
 '3PM',
 'FG3A',
 'FTM',
 'FTA',
 'OReb',
 'DReb',
 'REB',
 'Ast',
 'TO',
 

# Step 9: Generate 9 Man Rotation

In [21]:
# Helper function that calculates TO%
# Formula:
#     TO% = 100 * TOV / (FGA + 0.44 * FTA + TOV)

# Required columns:
#     FGA - Field goal attempts
#     FTA - Free throw attempts
#     TO  - Turnovers

# Returns:
#     pandas.Series containing TO%
def calculate_team_to_pct(df):
    denominator = df["FGA"] + 0.44 * df["FTA"] + df["TO"]

    return 100 * df["TO"] / denominator


In [22]:
# Helper function that Calculates the percentage of total points scored in the paint.
# Formula:
#     %PTS PITP = (Points in Paint / Total Points) * 100
# Required columns:
#     Pts_Paint - Points scored in the paint
#     Pts       - Total points scored
# Returns:
#     pandas.Series containing %PTS PITP
def calculate_pct_points_in_paint(df):
    return (df["Pts_Paint"] / df["Pts"]) * 100


In [23]:
# Helper fucntion that gets the current 9-man rotation for a team.
# Rules:
#     - Players with RemainingGames <= 0 are unavailable.
#     - Top 5 available players by Min are starters.
#     - Next 4 highest-minute available players are bench.
#     - Final 9 must contain at least 2 centers when possible.
def get_9_man_rotation(team_df):

    # Only use currently available players
    available = team_df[
        team_df["RemainingGames"] > 0
    ]

    if available.empty:
        return available

    # Minutes
    minutes = pd.to_numeric(
        available["Min"],
        errors="coerce"
    ).fillna(0)

    # Identify centers
    is_center = (
        available["Pos"]
        .fillna("")
        .astype(str)
        .str.upper()
        .str.contains("C")
    )

    # Sort by minutes
    sorted_indices = minutes.sort_values(
        ascending=False
    ).index

    # Initial rotation
    rotation_indices = list(
        sorted_indices[:9]
    )

    # Make sure we have 2 centers
    while (
        is_center.loc[rotation_indices].sum() < 2
    ):

        available_centers = [
            idx
            for idx in sorted_indices
            if idx not in rotation_indices
            and is_center.loc[idx]
        ]

        if not available_centers:
            break

        replacement = available_centers[0]

        non_centers = [
            idx
            for idx in rotation_indices
            if not is_center.loc[idx]
        ]

        if not non_centers:
            break

        remove_index = min(
            non_centers,
            key=lambda idx: minutes.loc[idx]
        )

        rotation_indices.remove(remove_index)
        rotation_indices.append(replacement)

    # Create rotation
    rotation = available.loc[
        rotation_indices
    ].copy()

    rotation = rotation.sort_values(
        "Min",
        ascending=False
    )

    # Starter / bench
    rotation["Rotation_Role"] = [
        "Starter" if i < 5 else "Bench"
        for i in range(len(rotation))
    ]

    return rotation.reset_index(drop=True)


In [24]:
# Helper function that normalizes minutes and stats for a 9-man rotation of a team to 240 minutes.
# Rules:
#     - Convert Min to numeric values.
#     - Calculate the current total minutes for the 9-man rotation.
#     - Scale each player's minutes proportionally so the total equals 240.
#     - Preserve the relative distribution of minutes between players.
# Returns: The normalized rotation as a copy of the original DataFrame.
# def normalize_rotation(rotation):
#     rotation = rotation.copy()

#     rotation["Min"] = pd.to_numeric(
#         rotation["Min"],
#         errors="coerce"
#     ).fillna(0)

#     # Preserve original minutes
#     rotation["OriginalMin"] = rotation["Min"]

#     total_minutes = rotation["Min"].sum()

#     if total_minutes <= 0:
#         return rotation

#     # Scale minutes
#     rotation["Min"] = (
#         rotation["Min"] / total_minutes
#     ) * 240

#     # Scale per-game stats to the new minutes
#     stat_columns = [
#         'Ast',
#         'DReb',
#         '3PM',
#         'OReb',
#         'FTA',
#         'FGA',
#         "TO",
#         'FTM',
#         'PTSOFFTO',
#         'Pts_Paint',
#         'Pts',
#         'Pts_Paint', 'Pts_MidRange', 'Pts_3PT'
#     ]
#     for stat in stat_columns:
#         rotation[stat] = (
#             rotation[stat] *
#             rotation["Min"] /
#             rotation["OriginalMin"]
#         )

#     return rotation

In [25]:
def normalize_rotation(rotation):
    rotation = rotation.copy()

    rotation["Min"] = pd.to_numeric(
        rotation["Min"],
        errors="coerce"
    ).fillna(0)

    rotation["OriginalMin"] = rotation["Min"]

    total_minutes = rotation["Min"].sum()

    if total_minutes <= 0:
        return rotation

    scale_factor = 240 / total_minutes

    print(
        f"Rotation minutes: {total_minutes:.2f} "
        f"-> 240 | Scale: {scale_factor:.3f}"
    )

    rotation["Min"] = rotation["Min"] * scale_factor

    stat_columns = [
        'Ast', 'DReb', '3PM', 'OReb',
        'FTA', 'FGA', 'TO', 'FTM',
        'PTSOFFTO', 'Pts_Paint', 'Pts',
        'Pts_MidRange', 'Pts_3PT'
    ]

    for stat in stat_columns:
        rotation[stat] = (
            rotation[stat]
            * rotation["Min"]
            / rotation["OriginalMin"]
        )

    return rotation


In [26]:
# Helper fucntion that subtracts 1 from GP for every player in the 9-man rotation.
# team_df: Mutable team roster.
# rotation:Current 9-man rotation.
def subtract_games_Played(team_df, rotation):
    player_ids = rotation["PlayerID"]
    mask = team_df["PlayerID"].isin(player_ids)
    team_df.loc[mask, "RemainingGames"] -= 1
    return team_df

# Step 10: Create Conferences/Standings

In [27]:
# Helper function that updates W/L records and winning percentage for a completed game.
def update_standings(standings, winner, loser):

    def find_conference(team):
        if team in standings["East"]:
            return "East"
        elif team in standings["West"]:
            return "West"
        else:
            raise ValueError(f"Unknown team abbreviation: {team}")

    winner_conf = find_conference(winner)
    loser_conf = find_conference(loser)

    # Update records
    standings[winner_conf][winner]["W"] += 1
    standings[loser_conf][loser]["L"] += 1

    # Recalculate winning percentages for any conference(s) touched
    for conf in {winner_conf, loser_conf}:
        for team, record in standings[conf].items():
            games = record["W"] + record["L"]
            record["W_Pct"] = record["W"] / games if games > 0 else 0.0

    return standings

In [28]:
# Helper function that creates initial NBA standings using TeamAbbv.
# Returns:
#     standings = {
#         "East": {...},
#         "West": {...}
#     }

# Each team starts with:
#     W = 0
#     L = 0
#     W_Pct = 0
def create_standings():

    eastern_teams = [
        "ATL",  # Atlanta Hawks
        "BOS",  # Boston Celtics
        "BKN",  # Brooklyn Nets
        "CHA",  # Charlotte Hornets
        "CHI",  # Chicago Bulls
        "CLE",  # Cleveland Cavaliers
        "DET",  # Detroit Pistons
        "IND",  # Indiana Pacers
        "MIA",  # Miami Heat
        "MIL",  # Milwaukee Bucks
        "NYK",  # New York Knicks
        "ORL",  # Orlando Magic
        "PHI",  # Philadelphia 76ers
        "TOR",  # Toronto Raptors
        "WAS",  # Washington Wizards
    ]

    western_teams = [
        "DAL",  # Dallas Mavericks
        "DEN",  # Denver Nuggets
        "GSW",  # Golden State Warriors
        "HOU",  # Houston Rockets
        "LAC",  # LA Clippers
        "LAL",  # Los Angeles Lakers
        "MEM",  # Memphis Grizzlies
        "MIN",  # Minnesota Timberwolves
        "NOP",  # New Orleans Pelicans
        "OKC",  # Oklahoma City Thunder
        "PHX",  # Phoenix Suns
        "POR",  # Portland Trail Blazers
        "SAC",  # Sacramento Kings
        "SAS",  # San Antonio Spurs
        "UTA",  # Utah Jazz
    ]

    standings = {
        "East": {},
        "West": {}
    }

    for team in eastern_teams:
        standings["East"][team] = {
            "W": 0,
            "L": 0,
            "W_Pct": 0.0
        }

    for team in western_teams:
        standings["West"][team] = {
            "W": 0,
            "L": 0,
            "W_Pct": 0.0
        }

    return standings


In [29]:
# Helper function that prints current NBA standings for both conferences.
def print_standings(standings):

    print("EASTERN CONFERENCE")

    east_sorted = sorted(
        standings["East"].items(),
        key=lambda x: x[1]["W_Pct"],
        reverse=True
    )

    print(
        f"{'Team':<6}"
        f"{'W':>5}"
        f"{'L':>5}"
        f"{'W_Pct':>10}"
    )

    for team, record in east_sorted:
        print(
            f"{team:<6}"
            f"{record['W']:>5}"
            f"{record['L']:>5}"
            f"{record['W_Pct']:>10.3f}"
        )

    print("WESTERN CONFERENCE")

    west_sorted = sorted(
        standings["West"].items(),
        key=lambda x: x[1]["W_Pct"],
        reverse=True
    )

    print(
        f"{'Team':<6}"
        f"{'W':>5}"
        f"{'L':>5}"
        f"{'W_Pct':>10}"
    )

    for team, record in west_sorted:
        print(
            f"{team:<6}"
            f"{record['W']:>5}"
            f"{record['L']:>5}"
            f"{record['W_Pct']:>10.3f}"
        )

In [30]:
# Helper function that compares the standing that the model obtained with the actual standings.
def compare_standings(model_standings, actual_standings):

    comparison = {
        "East": {},
        "West": {}
    }

    for conference in ["East", "West"]:

        for team, model_record in model_standings[conference].items():

            actual_record = actual_standings[conference][team]

            model_w = model_record["W"]
            model_l = model_record["L"]

            actual_w = actual_record["W"]
            actual_l = actual_record["L"]

            model_pct = model_record["W_Pct"]

            actual_games = actual_w + actual_l
            actual_pct = (
                actual_w / actual_games
                if actual_games > 0
                else 0.0
            )

            comparison[conference][team] = {
                "Model_W": model_w,
                "Actual_W": actual_w,
                "W_Diff": model_w - actual_w,

                "Model_L": model_l,
                "Actual_L": actual_l,
                "L_Diff": model_l - actual_l,

                "Model_W_Pct": model_pct,
                "Actual_W_Pct": actual_pct,

                "W_Pct_Diff": model_pct - actual_pct
            }

    return comparison

In [31]:
# Helper function that takes into account home-court advantage.
#     The boost is a logit-space shift; 0.20 corresponds to roughly a 50/50 game
#     becoming ~55/45 in the home team's favor (the NBA's long-run home edge).
def apply_home_advantage(home_prob, away_prob, boost=0.20):
    def logit(p):
        p = np.clip(p, 1e-6, 1 - 1e-6)
        return np.log(p / (1 - p))

    def sigmoid(x):
        return 1 / (1 + np.exp(-x))

    home_prob_adj = sigmoid(logit(home_prob) + boost)
    away_prob_adj = sigmoid(logit(away_prob) - boost)

    return home_prob_adj, away_prob_adj

Debugging

In [32]:
season = "21_22"
team = "BOS"

rotation = get_9_man_rotation(
    team_rosters[season][team]
)

rotation = rotation[
    [
        "PlayerName",
        "Pos",
        "GP",
        "RemainingGames",
        "Min",
        "Pts",
        'Ast', 
        'PTSOFFTO', 
        'TO', "FTA", 'FGA', # To get TO%
        'DReb', 'OReb',
        '3PM',
        'FTM',
        'FT_PCT',
        "Pts_Paint",
        "Pts_MidRange",
        "Pts_3PT",
        "Rotation_Role"
    ]
]

# Make sure minutes are numeric
rotation["Min"] = pd.to_numeric(
    rotation["Min"],
    errors="coerce"
).fillna(0)

# Primary position = first position listed
rotation["PrimaryPos"] = (
    rotation["Pos"]
    .fillna("")
    .astype(str)
    .str.upper()
    .str.split("-")
    .str[0]
    .str.strip()
)

# Get players by primary position
guards = rotation[rotation["PrimaryPos"] == "G"]
forwards = rotation[rotation["PrimaryPos"] == "F"]
centers = rotation[rotation["PrimaryPos"] == "C"]

# Minutes by primary position
guard_minutes = guards["Min"].sum()
forward_minutes = forwards["Min"].sum()
center_minutes = centers["Min"].sum()

total_minutes = rotation["Min"].sum()

display(
    rotation.drop(columns=["PrimaryPos"])
)

print(f"{team} 9-MAN ROTATION")

print("\nGUARDS")
for _, player in guards.iterrows():
    print(
        f"{player['PlayerName']:<25} "
        f"{player['Min']:>5.1f} min  "
        f"({player['Pos']})"
    )
print(f"Guard total: {guard_minutes:.1f} minutes")


print("\nFORWARDS")
for _, player in forwards.iterrows():
    print(
        f"{player['PlayerName']:<25} "
        f"{player['Min']:>5.1f} min  "
        f"({player['Pos']})"
    )
print(f"Forward total: {forward_minutes:.1f} minutes")


print("\nCENTERS")
for _, player in centers.iterrows():
    print(
        f"{player['PlayerName']:<25} "
        f"{player['Min']:>5.1f} min  "
        f"({player['Pos']})"
    )
print(f"Center total: {center_minutes:.1f} minutes")

print(f"TOTAL ROTATION MINUTES: {total_minutes:.1f}")
print(
    f"POSITION TOTAL: "
    f"{guard_minutes + forward_minutes + center_minutes:.1f}"
)


,PlayerName,Pos,GP,RemainingGames,Min,Pts,Ast,PTSOFFTO,TO,FTA,FGA,DReb,OReb,3PM,FTM,FT_PCT,Pts_Paint,Pts_MidRange,Pts_3PT,Rotation_Role
0,Jayson Tatum,F-G,76,76,35.933838,26.921053,4.394737,3.6,2.855263,6.171053,20.578947,6.894737,1.118421,3.026316,5.263158,0.853,10.052632,2.500000,9.118421,Starter
1,Jaylen Brown,G-F,66,66,33.643232,23.621212,3.500000,3.5,2.696970,4.818182,18.439394,5.318182,0.803030,2.515152,3.651515,0.758,9.878788,2.545455,7.545455,Starter
2,Marcus Smart,G,71,71,32.335798,12.126761,5.873239,1.7,2.239437,2.521127,10.112676,3.211268,0.591549,1.676056,2.000000,0.793,4.873239,0.253521,4.985915,Starter
3,Robert Williams III,C-F,61,61,29.579235,9.950820,1.983607,1.7,1.032787,1.475410,6.032787,5.737705,3.901639,0.000000,1.065574,0.722,8.754098,0.131148,0.000000,Starter
4,Derrick White,G,75,75,29.317356,13.226667,4.880000,1.6,1.586667,3.133333,10.666667,2.973333,0.506667,1.533333,2.706667,0.864,5.226667,0.693333,4.600000,Starter
5,Al Horford,C-F,69,69,29.054952,10.159420,3.362319,1.5,0.942029,1.376812,8.246377,6.115942,1.565217,1.289855,1.159420,0.842,4.463768,0.666667,3.869565,Bench
6,Grant Williams,F,77,77,24.351688,7.818182,1.012987,1.1,0.805195,1.233766,5.610390,2.727273,0.831169,1.376623,1.116883,0.905,2.493506,0.077922,4.129870,Bench
7,Daniel Theis,F-C,47,47,20.796170,8.170213,0.872340,1.0,0.978723,1.531915,6.191489,3.595745,1.255319,0.702128,1.042553,0.681,4.680851,0.340426,2.106383,Bench
8,Juwan Morgan,F,2,2,15.482500,2.500000,0.500000,0.0,0.000000,0.000000,1.500000,1.000000,1.000000,0.500000,0.000000,0.000,1.000000,0.000000,1.500000,Bench


BOS 9-MAN ROTATION

GUARDS
Jaylen Brown               33.6 min  (G-F)
Marcus Smart               32.3 min  (G)
Derrick White              29.3 min  (G)
Guard total: 95.3 minutes

FORWARDS
Jayson Tatum               35.9 min  (F-G)
Grant Williams             24.4 min  (F)
Daniel Theis               20.8 min  (F-C)
Juwan Morgan               15.5 min  (F)
Forward total: 96.6 minutes

CENTERS
Robert Williams III        29.6 min  (C-F)
Al Horford                 29.1 min  (C-F)
Center total: 58.6 minutes
TOTAL ROTATION MINUTES: 250.5
POSITION TOTAL: 250.5


# Step 10: Get schedule

In [33]:
# Picking the season
SEASONS = [
    # "21_22",
    "22_23",
    # "23_24",
    # "24_25",
    # "25_26",
]

Load Model

In [34]:
model = xgb.Booster()
model.load_model("/Users/ronanhwang/Downloads/Github/nba_2020s/best_model_Altogether.json")
print(model.feature_names)

# Debug
print("\nMODEL ATTRIBUTES:")
print(model.attributes())

print("\nMODEL CONFIG:")
print(model.save_config())


['Ast', 'OppTO%', 'PointsOffTO', 'DReb', '3PM', 'FTA', 'FT%', 'diff_FTM', 'diff_OReb', 'diff_%Points_InPaint', 'diff_OppPointsInPaint', 'diff_PointsOffTO']

MODEL ATTRIBUTES:
{'best_iteration': '199', 'best_score': '0.0829409206790008', 'scikit_learn': '{"_estimator_type": "classifier"}'}

MODEL CONFIG:
{"learner":{"generic_param":{"device":"cpu","fail_on_invalid_gpu_id":"0","n_jobs":"0","nthread":"0","random_state":"0","rng_state":"0 1 1812433255 1900727105 1208447044 2481403966 4042607538 337614300 3232553940 1018809052 3202401494 1775180719 3192392114 594215549 184016991 829906058 610491522 3879932251 3139825610 297902587 4075895579 2943625357 3530655617 1423771745 2135928312 2891506774 1066338622 135451537 933040465 2759011858 2273819758 3545703099 2516396728 1272276355 3172048492 3267256201 2332199830 1975469449 392443598 1132453229 2900699076 1998300999 3847713992 512669506 1227792182 1629110240 112303347 2142631694 3647635483 1715036585 2508091258 1355887243 1884998310 390636008

Debug

In [35]:
SEASON_TIPOFF = {
    "21_22": "2021-10-19",
    "22_23": "2022-10-18",
    "23_24": "2023-10-24",
    "24_25": "2024-10-22",
    "25_26": "2025-10-21",
}

def process_all_seasons(data, team_rosters, seasons):

    results = {}

    for season in seasons:

        print(f"Processing season: {season}")

        # Get season schedule
        schedule = data[season]["ScheduleLeagueV2"].copy()

        # Parse dates once, up front, as real datetimes (this was the root
        # cause of the missing-games bug: string comparisons against
        # gameDateTimeEst were unreliable)
        schedule["gameDateTimeEst"] = pd.to_datetime(schedule["gameDateTimeEst"], utc=True)
        tipoff = pd.Timestamp(SEASON_TIPOFF[season], tz="UTC")

        # Copy rosters because gp will be modified during simulation
        season_rosters = {team: df.copy() for team, df in team_rosters[season].items()}

        # Create fresh standings
        standings = create_standings()

        # Sort schedule chronologically
        schedule = (schedule.sort_values("gameDateTimeEst").reset_index(drop=True))

        # Debugging remove later
        # Track prediction accuracy (informational only, never halts the loop)
        correct_predictions = 0
        incorrect_predictions = 0
        # ADDED
        flipped_by_home_boost = 0        # <-- ADD THIS

        # Process games
        for _, game in schedule.iterrows():

            label = str(game["gameLabel"]).strip() if pd.notna(game["gameLabel"]) else ""
            series = str(game["seriesText"]).strip() if pd.notna(game["seriesText"]) else ""

            # Only process regular season games:
            # - exclude preseason (games before the season's real tipoff date)
            # - exclude anything with a non-blank gameLabel EXCEPT "Emirates NBA Cup"
            #   (playoffs, play-in, all-star, rising stars, etc.)
            # - exclude Emirates NBA Cup games played at a neutral site
            #   (semifinals + championship, i.e. Vegas) — group play/quarterfinals
            #   DO count toward the regular season and must be kept
            is_non_regular = (
                (game["gameDateTimeEst"] < tipoff)
                or (label != "" and label != "Emirates NBA Cup")
                or (label == "Emirates NBA Cup" and series == "Neutral Site")
            )
            if is_non_regular:
                continue

            # Debug this will pose an issue later when 
            # games are incomplete so try remove when code is fully complete and see if it still works
            # Only process completed games (accept Final, Final/OT, Final/OT2, etc.)
            if not str(game["gameStatusText"]).startswith("Final"):
                continue

            # Get teams
            home_team = game["homeTeam_teamTricode"]
            away_team = game["awayTeam_teamTricode"]

            # Defensive backstop: skip any game involving a team not in our roster data
            if home_team not in season_rosters or away_team not in season_rosters:
                print(f"Skipping game with unknown team(s): {away_team} @ {home_team}")
                continue

            # Get current rosters
            home_roster = season_rosters[home_team]
            away_roster = season_rosters[away_team]

            # Get current 9-man rotations
            home_rotation = get_9_man_rotation(home_roster)
            away_rotation = get_9_man_rotation(away_roster)

            # Normalize each rotation to 240 total minutes
            home_rotation = normalize_rotation(home_rotation)
            away_rotation = normalize_rotation(away_rotation)

            # Aggregate home team rotation
            homeTeam = home_rotation.agg({
                "Ast": "sum",
                "DReb": "sum",
                "3PM": "sum",
                "OReb": "sum",

                # Used for TO% / OppTO%
                "FTA": "sum", "FGA": "sum", "TO": "sum",

                # Used for FT% and diff_FTM
                "FTM": "sum",

                # Used for diff_PointsOffTO
                "PTSOFFTO": "sum",

                # Used for diff_OppPointsInPaint and diff_%Points_InPaint
                "Pts_Paint": "sum", "Pts": "sum",
            })

            # Aggregate away team rotation
            awayTeam = away_rotation.agg({
                "Ast": "sum",
                "DReb": "sum",
                "3PM": "sum",
                "OReb": "sum",

                "FTA": "sum", "FGA": "sum", "TO": "sum",
                "FTM": "sum",
                "PTSOFFTO": "sum",
                "Pts_Paint": "sum", "Pts": "sum",
            })

            # Each team's own TO% and %Points-in-paint (needed to build OppTO% and diff_%Points_InPaint)
            home_TO_pct = calculate_team_to_pct(homeTeam)
            away_TO_pct = calculate_team_to_pct(awayTeam)

            home_pct_pitp = calculate_pct_points_in_paint(homeTeam)
            away_pct_pitp = calculate_pct_points_in_paint(awayTeam)

            home_FT_pct = 100 * homeTeam["FTM"] / homeTeam["FTA"]
            away_FT_pct = 100 * awayTeam["FTM"] / awayTeam["FTA"]


            print("\n--- NORMALIZED TEAM STATS ---")
            print(f"{away_team}:")
            print(awayTeam)

            print(f"\n{home_team}:")
            print(homeTeam)


            # Home team feature vector
            home_features = pd.Series({
                "Ast": homeTeam["Ast"],
                "OppTO%": away_TO_pct,
                "PointsOffTO": homeTeam["PTSOFFTO"],   # relabeled here: PTSOFFTO -> PointsOffTO
                "DReb": homeTeam["DReb"],
                "3PM": homeTeam["3PM"],
                "FTA": homeTeam["FTA"],
                "FT%": home_FT_pct,
                "diff_FTM": homeTeam["FTM"] - awayTeam["FTM"],
                "diff_OReb": homeTeam["OReb"] - awayTeam["OReb"],
                "diff_%Points_InPaint": home_pct_pitp - away_pct_pitp,
                "diff_OppPointsInPaint": homeTeam["Pts_Paint"] - awayTeam["Pts_Paint"],
                "diff_PointsOffTO": homeTeam["PTSOFFTO"] - awayTeam["PTSOFFTO"],
            })

            # Away team feature vector
            away_features = pd.Series({
                "Ast": awayTeam["Ast"],
                "OppTO%": home_TO_pct,
                "PointsOffTO": awayTeam["PTSOFFTO"],   # relabeled here: PTSOFFTO -> PointsOffTO
                "DReb": awayTeam["DReb"],
                "3PM": awayTeam["3PM"],
                "FTA": awayTeam["FTA"],
                "FT%": away_FT_pct,
                "diff_FTM": awayTeam["FTM"] - homeTeam["FTM"],
                "diff_OReb": awayTeam["OReb"] - homeTeam["OReb"],
                "diff_%Points_InPaint": away_pct_pitp - home_pct_pitp,
                "diff_OppPointsInPaint": awayTeam["Pts_Paint"] - homeTeam["Pts_Paint"],
                "diff_PointsOffTO": awayTeam["PTSOFFTO"] - homeTeam["PTSOFFTO"],
            })

            # Model features
            features = [
                'Ast', 'OppTO%', 'PointsOffTO', 'DReb', '3PM', 'FTA', 'FT%',
                'diff_FTM', 'diff_OReb', 'diff_%Points_InPaint',
                'diff_OppPointsInPaint', 'diff_PointsOffTO'
            ]

            home_team_stats = home_features.to_frame().T[features]
            away_team_stats = away_features.to_frame().T[features]

            home_dmatrix = xgb.DMatrix(home_team_stats, feature_names=features)
            away_dmatrix = xgb.DMatrix(away_team_stats, feature_names=features)

            home_prediction = model.predict(home_dmatrix)[0]
            away_prediction = model.predict(away_dmatrix)[0]

             # Capture pre-boost predictions                    # <-- ADD THIS
            pre_boost_home_win = home_prediction >= 0.5         # <-- ADD THIS

            # Apply home-court advantage (model has no home/away feature of its own) ADDED
            home_prediction, away_prediction = apply_home_advantage(home_prediction, away_prediction)

            home_win = home_prediction >= 0.5
            away_win = away_prediction >= 0.5

            if home_win != pre_boost_home_win:                 # <-- ADD THIS
                flipped_by_home_boost += 1                      # <-- ADD THIS


            if home_win == away_win:
                print("MODEL DISAGREEMENT: BOTH TEAMS HAVE THE SAME RESULT")
                print("=" * 70)
                print(f"Season: {season}")
                print(f"Game: {away_team} @ {home_team}")
                print(f"\nHome prediction: {home_prediction:.4f} ({'WIN' if home_win else 'LOSS'})")
                print(f"Away prediction: {away_prediction:.4f} ({'WIN' if away_win else 'LOSS'})")

                if home_prediction > away_prediction:
                    model_winner, model_loser = home_team, away_team
                    print(f"Resolved: {home_team} favored (higher predicted probability)")
                elif away_prediction > home_prediction:
                    model_winner, model_loser = away_team, home_team
                    print(f"Resolved: {away_team} favored (higher predicted probability)")
                else:
                    model_winner, model_loser = home_team, away_team
                    print(f"Exact probability tie - defaulting to home team ({home_team})")

            else:
                if home_win:
                    model_winner, model_loser = home_team, away_team
                else:
                    model_winner, model_loser = away_team, home_team

            home_score = game["homeTeam_score"]
            away_score = game["awayTeam_score"]

            if home_score != away_score:
                actual_winner = home_team if home_score > away_score else away_team
                if model_winner == actual_winner:
                    correct_predictions += 1
                else:
                    incorrect_predictions += 1

            update_standings(standings, model_winner, model_loser)

            subtract_games_Played(home_roster, home_rotation)
            subtract_games_Played(away_roster, away_rotation)

        total = correct_predictions + incorrect_predictions
        accuracy = correct_predictions / total if total else float("nan")

        results[season] = {
            "team_rosters": season_rosters,
            "standings": standings,
            "correct_predictions": correct_predictions,
            "incorrect_predictions": incorrect_predictions,
            "flipped_by_home_boost": flipped_by_home_boost,   # <-- ADD THIS
            "accuracy": accuracy
        }

        print(f"\nFinished {season} — accuracy: {accuracy:.3f} ({correct_predictions}/{total})")
        print(f"FINAL STANDINGS — {season}")
        print_standings(standings)
        print(f"Games flipped by home-court boost: {flipped_by_home_boost}")   # <-- ADD THIS

        print(f"\nFinished {season} — accuracy: {accuracy:.3f} ({correct_predictions}/{total})")

    return results

In [37]:
SEASON_TIPOFF = {
    "21_22": "2021-10-19",
    "22_23": "2022-10-18",
    "23_24": "2023-10-24",
    "24_25": "2024-10-22",
    "25_26": "2025-10-21",
}

def process_all_seasons(data, team_rosters, seasons):

    results = {}

    for season in seasons:

        print(f"Processing season: {season}")

        # Get season schedule
        schedule = data[season]["ScheduleLeagueV2"].copy()

        # Parse dates once, up front
        schedule["gameDateTimeEst"] = pd.to_datetime(
            schedule["gameDateTimeEst"],
            utc=True
        )

        tipoff = pd.Timestamp(SEASON_TIPOFF[season], tz="UTC")

        # First calendar month of the season
        first_month = tipoff.month
        first_month_year = tipoff.year

        # Copy rosters because gp will be modified during simulation
        season_rosters = {
            team: df.copy()
            for team, df in team_rosters[season].items()
        }

        # Create fresh standings
        standings = create_standings()

        # Sort schedule chronologically
        schedule = (
            schedule
            .sort_values("gameDateTimeEst")
            .reset_index(drop=True)
        )

        # Track prediction accuracy
        correct_predictions = 0
        incorrect_predictions = 0

        # Track home-court flips
        flipped_by_home_boost = 0

        # Process games
        for _, game in schedule.iterrows():

            label = (
                str(game["gameLabel"]).strip()
                if pd.notna(game["gameLabel"])
                else ""
            )

            series = (
                str(game["seriesText"]).strip()
                if pd.notna(game["seriesText"])
                else ""
            )

            # Only process regular season games:
            # - exclude preseason
            # - exclude anything with a non-blank gameLabel
            #   EXCEPT "Emirates NBA Cup"
            # - exclude neutral-site NBA Cup games
            is_non_regular = (
                (game["gameDateTimeEst"] < tipoff)
                or (label != "" and label != "Emirates NBA Cup")
                or (
                    label == "Emirates NBA Cup"
                    and series == "Neutral Site"
                )
            )

            if is_non_regular:
                continue

            # Stop after first calendar month
            game_date = game["gameDateTimeEst"]

            if (
                game_date.month != first_month
                or game_date.year != first_month_year
            ):
                break

            # Only process completed games
            if not str(game["gameStatusText"]).startswith("Final"):
                continue

            # Get Teams
            home_team = game["homeTeam_teamTricode"]
            away_team = game["awayTeam_teamTricode"]

            # Defensive backstop
            if (
                home_team not in season_rosters
                or away_team not in season_rosters
            ):
                print(
                    f"Skipping game with unknown team(s): "
                    f"{away_team} @ {home_team}"
                )
                continue

            # Get current rosters
            home_roster = season_rosters[home_team]
            away_roster = season_rosters[away_team]

            # Get current 9 man rotation
            home_rotation = get_9_man_rotation(home_roster)
            away_rotation = get_9_man_rotation(away_roster)

            # Normalize each rotation to 240 total minutes
            home_rotation = normalize_rotation(home_rotation)
            away_rotation = normalize_rotation(away_rotation)

            # Aggregate home team stats
            homeTeam = home_rotation.agg({
                "Ast": "sum",
                "DReb": "sum",
                "3PM": "sum",
                "OReb": "sum",

                # Used for TO% / OppTO%
                "FTA": "sum",
                "FGA": "sum",
                "TO": "sum",

                # Used for FT% and diff_FTM
                "FTM": "sum",

                # Used for diff_PointsOffTO
                "PTSOFFTO": "sum",

                # Used for paint features
                "Pts_Paint": "sum",
                "Pts": "sum",
            })

            # Aggregate away team stats
            awayTeam = away_rotation.agg({
                "Ast": "sum",
                "DReb": "sum",
                "3PM": "sum",
                "OReb": "sum",

                "FTA": "sum",
                "FGA": "sum",
                "TO": "sum",

                "FTM": "sum",

                "PTSOFFTO": "sum",

                "Pts_Paint": "sum",
                "Pts": "sum",
            })

            # Calculate team feature
            home_TO_pct = calculate_team_to_pct(homeTeam)
            away_TO_pct = calculate_team_to_pct(awayTeam)

            home_pct_pitp = calculate_pct_points_in_paint(homeTeam)
            away_pct_pitp = calculate_pct_points_in_paint(awayTeam)

            home_FT_pct = (
                100 * homeTeam["FTM"] / homeTeam["FTA"]
            )

            away_FT_pct = (
                100 * awayTeam["FTM"] / awayTeam["FTA"]
            )

            # ========================================================
            # HOME TEAM FEATURES
            # ========================================================
            home_features = pd.Series({
                "Ast": homeTeam["Ast"],
                "OppTO%": away_TO_pct,
                "PointsOffTO": homeTeam["PTSOFFTO"],
                "DReb": homeTeam["DReb"],
                "3PM": homeTeam["3PM"],
                "FTA": homeTeam["FTA"],
                "FT%": home_FT_pct,
                "diff_FTM": (
                    homeTeam["FTM"]
                    - awayTeam["FTM"]
                ),
                "diff_OReb": (
                    homeTeam["OReb"]
                    - awayTeam["OReb"]
                ),
                "diff_%Points_InPaint": (
                    home_pct_pitp
                    - away_pct_pitp
                ),
                "diff_OppPointsInPaint": (
                    homeTeam["Pts_Paint"]
                    - awayTeam["Pts_Paint"]
                ),
                "diff_PointsOffTO": (
                    homeTeam["PTSOFFTO"]
                    - awayTeam["PTSOFFTO"]
                ),
            })

            # Away team features
            away_features = pd.Series({
                "Ast": awayTeam["Ast"],
                "OppTO%": home_TO_pct,
                "PointsOffTO": awayTeam["PTSOFFTO"],
                "DReb": awayTeam["DReb"],
                "3PM": awayTeam["3PM"],
                "FTA": awayTeam["FTA"],
                "FT%": away_FT_pct,
                "diff_FTM": (
                    awayTeam["FTM"]
                    - homeTeam["FTM"]
                ),
                "diff_OReb": (
                    awayTeam["OReb"]
                    - homeTeam["OReb"]
                ),
                "diff_%Points_InPaint": (
                    away_pct_pitp
                    - home_pct_pitp
                ),
                "diff_OppPointsInPaint": (
                    awayTeam["Pts_Paint"]
                    - homeTeam["Pts_Paint"]
                ),
                "diff_PointsOffTO": (
                    awayTeam["PTSOFFTO"]
                    - homeTeam["PTSOFFTO"]
                ),
            })

            # Model features
            features = [
                "Ast",
                "OppTO%",
                "PointsOffTO",
                "DReb",
                "3PM",
                "FTA",
                "FT%",
                "diff_FTM",
                "diff_OReb",
                "diff_%Points_InPaint",
                "diff_OppPointsInPaint",
                "diff_PointsOffTO",
            ]

            home_team_stats = (
                home_features
                .to_frame()
                .T[features]
            )

            away_team_stats = (
                away_features
                .to_frame()
                .T[features]
            )

            home_dmatrix = xgb.DMatrix(
                home_team_stats,
                feature_names=features
            )

            away_dmatrix = xgb.DMatrix(
                away_team_stats,
                feature_names=features
            )

            # Model Predictions
            home_prediction = float(
                model.predict(home_dmatrix)[0]
            )

            away_prediction = float(
                model.predict(away_dmatrix)[0]
            )


            print(f"MODEL INPUT DEBUG: {away_team} @ {home_team}")

            print(f"\n{home_team} FEATURES:")
            for feature in features:
                print(f"  {feature:30s}: {home_team_stats.iloc[0][feature]:.6f}")

            print(f"\n{away_team} FEATURES:")
            for feature in features:
                print(f"  {feature:30s}: {away_team_stats.iloc[0][feature]:.6f}")

            print("\nRAW MODEL PREDICTIONS:")
            print(f"  {home_team}: {home_prediction:.6f}")
            print(f"  {away_team}: {away_prediction:.6f}")

            # Capture pre-boost winner
            pre_boost_winner = (
                home_team
                if home_prediction > away_prediction
                else away_team
            )

            # Home Court Advantage
            home_prediction, away_prediction = apply_home_advantage(
                home_prediction,
                away_prediction
            )

            # Determine winner after home-court advantage
            post_boost_winner = (
                home_team
                if home_prediction > away_prediction
                else away_team
            )

            if pre_boost_winner != post_boost_winner:
                flipped_by_home_boost += 1

            # Only use higher prediction to determine winner
            if home_prediction > away_prediction:
                model_winner = home_team
                model_loser = away_team
            else:
                model_winner = away_team
                model_loser = home_team

            # Actual Result
            home_score = game["homeTeam_score"]
            away_score = game["awayTeam_score"]

            print("\nACTUAL RESULT:")
            print(f"  {home_team}: {home_score}")
            print(f"  {away_team}: {away_score}")

            if home_score != away_score:

                actual_winner = (
                    home_team
                    if home_score > away_score
                    else away_team
                )

                print(f"  Actual Winner: {actual_winner}")
                print(f"  Model Winner:  {model_winner}")

                if model_winner == actual_winner:
                    print("  Prediction: CORRECT")
                    correct_predictions += 1
                else:
                    print("  Prediction: INCORRECT")
                    incorrect_predictions += 1

            # Update Standings and Rosters
            update_standings(
                standings,
                model_winner,
                model_loser
            )

            subtract_games_Played(
                home_roster,
                home_rotation
            )

            subtract_games_Played(
                away_roster,
                away_rotation
            )

        # First Month Accuracy
        total = correct_predictions + incorrect_predictions

        accuracy = (
            correct_predictions / total
            if total
            else float("nan")
        )

        results[season] = {
            "team_rosters": season_rosters,
            "standings": standings,
            "correct_predictions": correct_predictions,
            "incorrect_predictions": incorrect_predictions,
            "flipped_by_home_boost": flipped_by_home_boost,
            "accuracy": accuracy,
        }

        print(
            f"\nFinished first month {season} — "
            f"accuracy: {accuracy:.3f} "
            f"({correct_predictions}/{total})"
        )

        print(
            f"Games flipped by home-court boost: "
            f"{flipped_by_home_boost}"
        )

    return results

# Step 11: Check

In [38]:
results = process_all_seasons(
    data,
    team_rosters,
    SEASONS
)

Processing season: 22_23
Rotation minutes: 255.24 -> 240 | Scale: 0.940
Rotation minutes: 264.79 -> 240 | Scale: 0.906
MODEL INPUT DEBUG: PHI @ BOS

BOS FEATURES:
  Ast                           : 27.209211
  OppTO%                        : 12.174989
  PointsOffTO                   : 15.984937
  DReb                          : 35.994731
  3PM                           : 15.849376
  FTA                           : 22.400968
  FT%                           : 80.898275
  diff_FTM                      : -3.487133
  diff_OReb                     : 1.013938
  diff_%Points_InPaint          : 2.732926
  diff_OppPointsInPaint         : 0.601848
  diff_PointsOffTO              : -2.505478

PHI FEATURES:
  Ast                           : 28.093529
  OppTO%                        : 11.518771
  PointsOffTO                   : 18.490414
  DReb                          : 31.349883
  3PM                           : 15.061410
  FTA                           : 26.183018
  FT%                           :

In [37]:
nba_standings = {
    "21_22": {
        "west": [
            {"teamAbbv": "PHX", "W": 64, "L": 18, "W_Pct": 0.780},
            {"teamAbbv": "MEM", "W": 56, "L": 26, "W_Pct": 0.683},
            {"teamAbbv": "GSW", "W": 53, "L": 29, "W_Pct": 0.646},
            {"teamAbbv": "DAL", "W": 52, "L": 30, "W_Pct": 0.634},
            {"teamAbbv": "UTA", "W": 49, "L": 33, "W_Pct": 0.598},
            {"teamAbbv": "DEN", "W": 48, "L": 34, "W_Pct": 0.585},
            {"teamAbbv": "MIN", "W": 46, "L": 36, "W_Pct": 0.561},
            {"teamAbbv": "LAC", "W": 42, "L": 40, "W_Pct": 0.512},
            {"teamAbbv": "NOP", "W": 36, "L": 46, "W_Pct": 0.439},
            {"teamAbbv": "SAS", "W": 34, "L": 48, "W_Pct": 0.415},
            {"teamAbbv": "LAL", "W": 33, "L": 49, "W_Pct": 0.402},
            {"teamAbbv": "SAC", "W": 30, "L": 52, "W_Pct": 0.366},
            {"teamAbbv": "POR", "W": 27, "L": 55, "W_Pct": 0.329},
            {"teamAbbv": "OKC", "W": 24, "L": 58, "W_Pct": 0.293},
            {"teamAbbv": "HOU", "W": 20, "L": 62, "W_Pct": 0.244},
        ],
        "east": [
            {"teamAbbv": "MIA", "W": 53, "L": 29, "W_Pct": 0.646},
            {"teamAbbv": "BOS", "W": 51, "L": 31, "W_Pct": 0.622},
            {"teamAbbv": "MIL", "W": 51, "L": 31, "W_Pct": 0.622},
            {"teamAbbv": "PHI", "W": 51, "L": 31, "W_Pct": 0.622},
            {"teamAbbv": "TOR", "W": 48, "L": 34, "W_Pct": 0.585},
            {"teamAbbv": "CHI", "W": 46, "L": 36, "W_Pct": 0.561},
            {"teamAbbv": "BKN", "W": 44, "L": 38, "W_Pct": 0.537},
            {"teamAbbv": "CLE", "W": 44, "L": 38, "W_Pct": 0.537},
            {"teamAbbv": "ATL", "W": 43, "L": 39, "W_Pct": 0.524},
            {"teamAbbv": "CHA", "W": 43, "L": 39, "W_Pct": 0.524},
            {"teamAbbv": "NYK", "W": 37, "L": 45, "W_Pct": 0.451},
            {"teamAbbv": "WAS", "W": 35, "L": 47, "W_Pct": 0.427},
            {"teamAbbv": "IND", "W": 25, "L": 57, "W_Pct": 0.305},
            {"teamAbbv": "DET", "W": 23, "L": 59, "W_Pct": 0.280},
            {"teamAbbv": "ORL", "W": 22, "L": 60, "W_Pct": 0.268},
        ],
    },

    "22_23": {
        "east": [
            {"teamAbbv": "MIL", "W": 58, "L": 24, "W_Pct": 0.707},
            {"teamAbbv": "BOS", "W": 57, "L": 25, "W_Pct": 0.695},
            {"teamAbbv": "PHI", "W": 54, "L": 28, "W_Pct": 0.659},
            {"teamAbbv": "CLE", "W": 51, "L": 31, "W_Pct": 0.622},
            {"teamAbbv": "NYK", "W": 47, "L": 35, "W_Pct": 0.573},
            {"teamAbbv": "BKN", "W": 45, "L": 37, "W_Pct": 0.549},
            {"teamAbbv": "MIA", "W": 44, "L": 38, "W_Pct": 0.537},
            {"teamAbbv": "ATL", "W": 41, "L": 41, "W_Pct": 0.500},
            {"teamAbbv": "TOR", "W": 41, "L": 41, "W_Pct": 0.500},
            {"teamAbbv": "CHI", "W": 40, "L": 42, "W_Pct": 0.488},
            {"teamAbbv": "IND", "W": 35, "L": 47, "W_Pct": 0.427},
            {"teamAbbv": "WAS", "W": 35, "L": 47, "W_Pct": 0.427},
            {"teamAbbv": "ORL", "W": 34, "L": 48, "W_Pct": 0.415},
            {"teamAbbv": "CHA", "W": 27, "L": 55, "W_Pct": 0.329},
            {"teamAbbv": "DET", "W": 17, "L": 65, "W_Pct": 0.207},
        ],
        "west": [
            {"teamAbbv": "DEN", "W": 53, "L": 29, "W_Pct": 0.646},
            {"teamAbbv": "MEM", "W": 51, "L": 31, "W_Pct": 0.622},
            {"teamAbbv": "SAC", "W": 48, "L": 34, "W_Pct": 0.585},
            {"teamAbbv": "PHX", "W": 45, "L": 37, "W_Pct": 0.549},
            {"teamAbbv": "LAC", "W": 44, "L": 38, "W_Pct": 0.537},
            {"teamAbbv": "GSW", "W": 44, "L": 38, "W_Pct": 0.537},
            {"teamAbbv": "LAL", "W": 43, "L": 39, "W_Pct": 0.524},
            {"teamAbbv": "MIN", "W": 42, "L": 40, "W_Pct": 0.512},
            {"teamAbbv": "NOP", "W": 42, "L": 40, "W_Pct": 0.512},
            {"teamAbbv": "OKC", "W": 40, "L": 42, "W_Pct": 0.488},
            {"teamAbbv": "DAL", "W": 38, "L": 44, "W_Pct": 0.463},
            {"teamAbbv": "UTA", "W": 37, "L": 45, "W_Pct": 0.451},
            {"teamAbbv": "POR", "W": 33, "L": 49, "W_Pct": 0.402},
            {"teamAbbv": "HOU", "W": 22, "L": 60, "W_Pct": 0.268},
            {"teamAbbv": "SAS", "W": 22, "L": 60, "W_Pct": 0.268},
        ],
    },

    "23_24": {
        "east": [
            {"teamAbbv": "BOS", "W": 64, "L": 18, "W_Pct": 0.780},
            {"teamAbbv": "NYK", "W": 50, "L": 32, "W_Pct": 0.610},
            {"teamAbbv": "MIL", "W": 49, "L": 33, "W_Pct": 0.598},
            {"teamAbbv": "CLE", "W": 48, "L": 34, "W_Pct": 0.585},
            {"teamAbbv": "ORL", "W": 47, "L": 35, "W_Pct": 0.573},
            {"teamAbbv": "IND", "W": 47, "L": 35, "W_Pct": 0.573},
            {"teamAbbv": "PHI", "W": 47, "L": 35, "W_Pct": 0.573},
            {"teamAbbv": "MIA", "W": 46, "L": 36, "W_Pct": 0.561},
            {"teamAbbv": "CHI", "W": 39, "L": 43, "W_Pct": 0.476},
            {"teamAbbv": "ATL", "W": 36, "L": 46, "W_Pct": 0.439},
            {"teamAbbv": "BKN", "W": 32, "L": 50, "W_Pct": 0.390},
            {"teamAbbv": "TOR", "W": 25, "L": 57, "W_Pct": 0.305},
            {"teamAbbv": "CHA", "W": 21, "L": 61, "W_Pct": 0.256},
            {"teamAbbv": "WAS", "W": 15, "L": 67, "W_Pct": 0.183},
            {"teamAbbv": "DET", "W": 14, "L": 68, "W_Pct": 0.171},
        ],
        "west": [
            {"teamAbbv": "OKC", "W": 57, "L": 25, "W_Pct": 0.695},
            {"teamAbbv": "DEN", "W": 57, "L": 25, "W_Pct": 0.695},
            {"teamAbbv": "MIN", "W": 56, "L": 26, "W_Pct": 0.683},
            {"teamAbbv": "LAC", "W": 51, "L": 31, "W_Pct": 0.622},
            {"teamAbbv": "DAL", "W": 50, "L": 32, "W_Pct": 0.610},
            {"teamAbbv": "PHX", "W": 49, "L": 33, "W_Pct": 0.598},
            {"teamAbbv": "NOP", "W": 49, "L": 33, "W_Pct": 0.598},
            {"teamAbbv": "LAL", "W": 47, "L": 35, "W_Pct": 0.573},
            {"teamAbbv": "SAC", "W": 46, "L": 36, "W_Pct": 0.561},
            {"teamAbbv": "GSW", "W": 46, "L": 36, "W_Pct": 0.561},
            {"teamAbbv": "HOU", "W": 41, "L": 41, "W_Pct": 0.500},
            {"teamAbbv": "UTA", "W": 31, "L": 51, "W_Pct": 0.378},
            {"teamAbbv": "MEM", "W": 27, "L": 55, "W_Pct": 0.329},
            {"teamAbbv": "SAS", "W": 22, "L": 60, "W_Pct": 0.268},
            {"teamAbbv": "POR", "W": 21, "L": 61, "W_Pct": 0.256},
        ],
    },

    "24_25": {
        "east": [
            {"teamAbbv": "CLE", "W": 64, "L": 18, "W_Pct": 0.780},
            {"teamAbbv": "BOS", "W": 61, "L": 21, "W_Pct": 0.744},
            {"teamAbbv": "NYK", "W": 51, "L": 31, "W_Pct": 0.622},
            {"teamAbbv": "IND", "W": 50, "L": 32, "W_Pct": 0.610},
            {"teamAbbv": "MIL", "W": 48, "L": 34, "W_Pct": 0.585},
            {"teamAbbv": "DET", "W": 44, "L": 38, "W_Pct": 0.537},
            {"teamAbbv": "ORL", "W": 41, "L": 41, "W_Pct": 0.500},
            {"teamAbbv": "ATL", "W": 40, "L": 42, "W_Pct": 0.488},
            {"teamAbbv": "CHI", "W": 39, "L": 43, "W_Pct": 0.476},
            {"teamAbbv": "MIA", "W": 37, "L": 45, "W_Pct": 0.451},
            {"teamAbbv": "TOR", "W": 30, "L": 52, "W_Pct": 0.366},
            {"teamAbbv": "BKN", "W": 26, "L": 56, "W_Pct": 0.317},
            {"teamAbbv": "PHI", "W": 24, "L": 58, "W_Pct": 0.293},
            {"teamAbbv": "CHA", "W": 19, "L": 63, "W_Pct": 0.232},
            {"teamAbbv": "WAS", "W": 18, "L": 64, "W_Pct": 0.220},
        ],
        "west": [
            {"teamAbbv": "OKC", "W": 68, "L": 14, "W_Pct": 0.829},
            {"teamAbbv": "HOU", "W": 52, "L": 30, "W_Pct": 0.634},
            {"teamAbbv": "LAL", "W": 50, "L": 32, "W_Pct": 0.610},
            {"teamAbbv": "DEN", "W": 50, "L": 32, "W_Pct": 0.610},
            {"teamAbbv": "LAC", "W": 50, "L": 32, "W_Pct": 0.610},
            {"teamAbbv": "MIN", "W": 49, "L": 33, "W_Pct": 0.598},
            {"teamAbbv": "GSW", "W": 48, "L": 34, "W_Pct": 0.585},
            {"teamAbbv": "MEM", "W": 48, "L": 34, "W_Pct": 0.585},
            {"teamAbbv": "SAC", "W": 40, "L": 42, "W_Pct": 0.488},
            {"teamAbbv": "DAL", "W": 39, "L": 43, "W_Pct": 0.476},
            {"teamAbbv": "PHX", "W": 36, "L": 46, "W_Pct": 0.439},
            {"teamAbbv": "POR", "W": 36, "L": 46, "W_Pct": 0.439},
            {"teamAbbv": "SAS", "W": 34, "L": 48, "W_Pct": 0.415},
            {"teamAbbv": "NOP", "W": 21, "L": 61, "W_Pct": 0.256},
            {"teamAbbv": "UTA", "W": 17, "L": 65, "W_Pct": 0.207},
        ],
    },

    "25_26": {
        "east": [
            {"teamAbbv": "DET", "W": 60, "L": 22, "W_Pct": 0.732},
            {"teamAbbv": "BOS", "W": 56, "L": 26, "W_Pct": 0.683},
            {"teamAbbv": "NYK", "W": 53, "L": 29, "W_Pct": 0.646},
            {"teamAbbv": "CLE", "W": 52, "L": 30, "W_Pct": 0.634},
            {"teamAbbv": "TOR", "W": 46, "L": 36, "W_Pct": 0.561},
            {"teamAbbv": "ATL", "W": 46, "L": 36, "W_Pct": 0.561},
            {"teamAbbv": "PHI", "W": 45, "L": 37, "W_Pct": 0.549},
            {"teamAbbv": "ORL", "W": 45, "L": 37, "W_Pct": 0.549},
            {"teamAbbv": "CHA", "W": 44, "L": 38, "W_Pct": 0.537},
            {"teamAbbv": "MIA", "W": 43, "L": 39, "W_Pct": 0.524},
            {"teamAbbv": "MIL", "W": 32, "L": 50, "W_Pct": 0.390},
            {"teamAbbv": "CHI", "W": 31, "L": 51, "W_Pct": 0.378},
            {"teamAbbv": "BKN", "W": 20, "L": 62, "W_Pct": 0.244},
            {"teamAbbv": "IND", "W": 19, "L": 63, "W_Pct": 0.232},
            {"teamAbbv": "WAS", "W": 17, "L": 65, "W_Pct": 0.207},
        ],
        "west": [
            {"teamAbbv": "OKC", "W": 64, "L": 18, "W_Pct": 0.780},
            {"teamAbbv": "SAS", "W": 62, "L": 20, "W_Pct": 0.756},
            {"teamAbbv": "DEN", "W": 54, "L": 28, "W_Pct": 0.659},
            {"teamAbbv": "LAL", "W": 53, "L": 29, "W_Pct": 0.646},
            {"teamAbbv": "HOU", "W": 52, "L": 30, "W_Pct": 0.634},
            {"teamAbbv": "MIN", "W": 49, "L": 33, "W_Pct": 0.598},
            {"teamAbbv": "PHX", "W": 45, "L": 37, "W_Pct": 0.549},
            {"teamAbbv": "POR", "W": 42, "L": 40, "W_Pct": 0.512},
            {"teamAbbv": "LAC", "W": 42, "L": 40, "W_Pct": 0.512},
            {"teamAbbv": "GSW", "W": 37, "L": 45, "W_Pct": 0.451},
            {"teamAbbv": "NOP", "W": 26, "L": 56, "W_Pct": 0.317},
            {"teamAbbv": "DAL", "W": 26, "L": 56, "W_Pct": 0.317},
            {"teamAbbv": "MEM", "W": 25, "L": 57, "W_Pct": 0.305},
            {"teamAbbv": "SAC", "W": 22, "L": 60, "W_Pct": 0.268},
            {"teamAbbv": "UTA", "W": 22, "L": 60, "W_Pct": 0.268},
        ],
    },
}


In [38]:
vegas_odds = {
    "21_22": {
        "BKN": {"odds": 240, "wl_ou": 56.5},
        "LAL": {"odds": 400, "wl_ou": 52.5},
        "MIL": {"odds": 900, "wl_ou": 54.5},
        "GSW": {"odds": 900, "wl_ou": 48.5},
        "PHX": {"odds": 1500, "wl_ou": 51.5},
        "UTA": {"odds": 1600, "wl_ou": 52.5},
        "PHI": {"odds": 1600, "wl_ou": 50.5},
        "LAC": {"odds": 1600, "wl_ou": 45.5},
        "DEN": {"odds": 2200, "wl_ou": 47.5},
        "MIA": {"odds": 2500, "wl_ou": 48.5},
        "DAL": {"odds": 3000, "wl_ou": 48.5},
        "ATL": {"odds": 3500, "wl_ou": 47.5},
        "BOS": {"odds": 4000, "wl_ou": 45.5},
        "POR": {"odds": 5000, "wl_ou": 44.5},
        "CHI": {"odds": 6600, "wl_ou": 42.5},
        "NYK": {"odds": 8000, "wl_ou": 41.5},
        "MEM": {"odds": 10000, "wl_ou": 41.5},
        "IND": {"odds": 10000, "wl_ou": 42.5},
        "NOP": {"odds": 10000, "wl_ou": 39.5},
        "CHA": {"odds": 10000, "wl_ou": 38.5},
        "WAS": {"odds": 15000, "wl_ou": 33.5},
        "TOR": {"odds": 15000, "wl_ou": 35.5},
        "SAC": {"odds": 25000, "wl_ou": 36.5},
        "MIN": {"odds": 25000, "wl_ou": 35.5},
        "SAS": {"odds": 30000, "wl_ou": 28.5},
        "CLE": {"odds": 50000, "wl_ou": 26.5},
        "ORL": {"odds": 50000, "wl_ou": 22.5},
        "OKC": {"odds": 50000, "wl_ou": 23.5},
        "DET": {"odds": 50000, "wl_ou": 24.5},
        "HOU": {"odds": 50000, "wl_ou": 24.5},
    },

    "22_23": {
        "BOS": {"odds": 500, "wl_ou": 54.5},
        "GSW": {"odds": 600, "wl_ou": 52.5},
        "MIL": {"odds": 650, "wl_ou": 53.5},
        "BKN": {"odds": 700, "wl_ou": 50.5},
        "LAC": {"odds": 700, "wl_ou": 52.5},
        "PHX": {"odds": 1000, "wl_ou": 52.5},
        "PHI": {"odds": 1400, "wl_ou": 50.5},
        "MIA": {"odds": 1600, "wl_ou": 49.5},
        "DEN": {"odds": 1800, "wl_ou": 51.5},
        "LAL": {"odds": 1800, "wl_ou": 44.5},
        "MEM": {"odds": 2000, "wl_ou": 49.5},
        "DAL": {"odds": 2500, "wl_ou": 48.5},
        "MIN": {"odds": 2800, "wl_ou": 49.5},
        "CLE": {"odds": 3000, "wl_ou": 46.5},
        "TOR": {"odds": 4000, "wl_ou": 46.5},
        "CHI": {"odds": 4000, "wl_ou": 41.5},
        "NOP": {"odds": 4000, "wl_ou": 45.5},
        "ATL": {"odds": 5000, "wl_ou": 46.5},
        "NYK": {"odds": 6600, "wl_ou": 38.5},
        "POR": {"odds": 8000, "wl_ou": 39.5},
        "CHA": {"odds": 15000, "wl_ou": 34.5},
        "WAS": {"odds": 25000, "wl_ou": 35.5},
        "DET": {"odds": 50000, "wl_ou": 29.5},
        "OKC": {"odds": 50000, "wl_ou": 23.5},
        "ORL": {"odds": 50000, "wl_ou": 26.5},
        "IND": {"odds": 50000, "wl_ou": 24.5},
        "SAC": {"odds": 50000, "wl_ou": 34.5},
        "SAS": {"odds": 50000, "wl_ou": 22.5},
        "HOU": {"odds": 50000, "wl_ou": 23.5},
        "UTA": {"odds": 50000, "wl_ou": 23.5},
    },

    "23_24": {
        "BOS": {"odds": 450, "wl_ou": 54.5},
        "DEN": {"odds": 450, "wl_ou": 52.5},
        "PHX": {"odds": 600, "wl_ou": 51.5},
        "MIL": {"odds": 600, "wl_ou": 54.5},
        "GSW": {"odds": 1200, "wl_ou": 48.5},
        "LAL": {"odds": 1500, "wl_ou": 47.5},
        "MIA": {"odds": 1600, "wl_ou": 45.5},
        "PHI": {"odds": 1800, "wl_ou": 48.5},
        "LAC": {"odds": 2200, "wl_ou": 46.5},
        "MEM": {"odds": 2200, "wl_ou": 45.5},
        "DAL": {"odds": 2500, "wl_ou": 45.5},
        "CLE": {"odds": 2500, "wl_ou": 50.5},
        "SAC": {"odds": 3500, "wl_ou": 44.5},
        "NOP": {"odds": 4000, "wl_ou": 44.5},
        "NYK": {"odds": 5000, "wl_ou": 45.5},
        "MIN": {"odds": 6600, "wl_ou": 44.5},
        "ATL": {"odds": 8000, "wl_ou": 42.5},
        "TOR": {"odds": 8000, "wl_ou": 36.5},
        "OKC": {"odds": 10000, "wl_ou": 44.5},
        "CHI": {"odds": 12500, "wl_ou": 37.5},
        "BKN": {"odds": 12500, "wl_ou": 37.5},
        "POR": {"odds": 12500, "wl_ou": 28.5},
        "SAS": {"odds": 15000, "wl_ou": 28.5},
        "UTA": {"odds": 25000, "wl_ou": 35.5},
        "ORL": {"odds": 25000, "wl_ou": 37.5},
        "IND": {"odds": 25000, "wl_ou": 38.5},
        "HOU": {"odds": 25000, "wl_ou": 31.5},
        "WAS": {"odds": 50000, "wl_ou": 24.5},
        "CHA": {"odds": 50000, "wl_ou": 31.5},
        "DET": {"odds": 50000, "wl_ou": 28.5},
    },

    "24_25": {
        "BOS": {"odds": 325, "wl_ou": 58.5},
        "OKC": {"odds": 675, "wl_ou": 57.5},
        "NYK": {"odds": 700, "wl_ou": 53.5},
        "PHI": {"odds": 1000, "wl_ou": 50.5},
        "DAL": {"odds": 1100, "wl_ou": 49.5},
        "DEN": {"odds": 1100, "wl_ou": 51.5},
        "MIN": {"odds": 1100, "wl_ou": 51.5},
        "MIL": {"odds": 1400, "wl_ou": 49.5},
        "PHX": {"odds": 1500, "wl_ou": 48.5},
        "CLE": {"odds": 3500, "wl_ou": 48.5},
        "MEM": {"odds": 3500, "wl_ou": 47.5},
        "ORL": {"odds": 4000, "wl_ou": 47.5},
        "GSW": {"odds": 4000, "wl_ou": 43.5},
        "LAL": {"odds": 4000, "wl_ou": 43.5},
        "MIA": {"odds": 5000, "wl_ou": 44.5},
        "NOP": {"odds": 5000, "wl_ou": 46.5},
        "IND": {"odds": 6600, "wl_ou": 46.5},
        "SAC": {"odds": 6600, "wl_ou": 46.5},
        "HOU": {"odds": 8000, "wl_ou": 43.5},
        "LAC": {"odds": 12500, "wl_ou": 35.5},
        "SAS": {"odds": 15000, "wl_ou": 35.5},
        "ATL": {"odds": 50000, "wl_ou": 36.5},
        "UTA": {"odds": 50000, "wl_ou": 28.5},
        "DET": {"odds": 100000, "wl_ou": 25.5},
        "CHA": {"odds": 100000, "wl_ou": 30.5},
        "CHI": {"odds": 100000, "wl_ou": 28.5},
        "BKN": {"odds": 100000, "wl_ou": 19.5},
        "POR": {"odds": 100000, "wl_ou": 21.5},
        "TOR": {"odds": 100000, "wl_ou": 29.5},
        "WAS": {"odds": 100000, "wl_ou": 20.5},
    },

    "25_26": {
        "OKC": {"odds": 240, "wl_ou": 62.5},
        "DEN": {"odds": 550, "wl_ou": 53.5},
        "CLE": {"odds": 750, "wl_ou": 56.5},
        "NYK": {"odds": 900, "wl_ou": 53.5},
        "MIN": {"odds": 1300, "wl_ou": 49.5},
        "HOU": {"odds": 1400, "wl_ou": 52.5},
        "LAL": {"odds": 1600, "wl_ou": 46.5},
        "LAC": {"odds": 1800, "wl_ou": 49.5},
        "ORL": {"odds": 1800, "wl_ou": 51.5},
        "GSW": {"odds": 2500, "wl_ou": 47.5},
        "DET": {"odds": 3300, "wl_ou": 46.5},
        "DAL": {"odds": 3500, "wl_ou": 41.5},
        "PHI": {"odds": 4000, "wl_ou": 43.5},
        "ATL": {"odds": 4000, "wl_ou": 46.5},
        "MIL": {"odds": 5500, "wl_ou": 43.5},
        "BOS": {"odds": 6000, "wl_ou": 41.5},
        "SAS": {"odds": 6600, "wl_ou": 44.5},
        "IND": {"odds": 10000, "wl_ou": 37.5},
        "TOR": {"odds": 10000, "wl_ou": 39.5},
        "MEM": {"odds": 12500, "wl_ou": 39.5},
        "MIA": {"odds": 20000, "wl_ou": 37.5},
        "NOP": {"odds": 30000, "wl_ou": 30.5},
        "PHX": {"odds": 50000, "wl_ou": 30.5},
        "POR": {"odds": 50000, "wl_ou": 35.5},
        "SAC": {"odds": 50000, "wl_ou": 35.5},
        "CHI": {"odds": 50000, "wl_ou": 33.5},
        "CHA": {"odds": 100000, "wl_ou": 27.5},
        "BKN": {"odds": 100000, "wl_ou": 19.5},
        "UTA": {"odds": 100000, "wl_ou": 18.5},
        "WAS": {"odds": 100000, "wl_ou": 20.5},
    },
}

In [39]:
# Create dataframe to compare standings 22_23

season = "22_23"

# WEST


# Model standings
model_west = (
    pd.DataFrame(results[season]["standings"]["West"])
    .T
    .reset_index()
    .rename(columns={
        "index": "Team",
        "W_Pct": "W%"
    })
)

# Keep only the relevant columns
model_west = model_west[["Team", "W"]].rename(
    columns={"W": "Model Wins"}
)

# Actual standings
actual_west = (
    pd.DataFrame(nba_standings[season]["west"])
    .rename(columns={
        "teamAbbv": "Team",
        "W_Pct": "W%"
    })
)

actual_west = actual_west[["Team", "W"]].rename(
    columns={"W": "Actual Wins"}
)

# Vegas
vegas_west = (
    pd.DataFrame.from_dict(
        vegas_odds[season],
        orient="index"
    )
    .reset_index()
    .rename(columns={
        "index": "Team",
        "wl_ou": "W-L O/U"
    })
)

vegas_west = vegas_west[["Team", "W-L O/U"]]

# Merge
west_final = (
    model_west
    .merge(vegas_west, on="Team", how="left")
    .merge(actual_west, on="Team", how="left")
)

# Differences
west_final["Model - Vegas"] = (
    west_final["Model Wins"] - west_final["W-L O/U"]
)

west_final["Actual - Vegas"] = (
    west_final["Actual Wins"] - west_final["W-L O/U"]
)


# Final Column Order
west_final = west_final[
    [
        "Team",
        "Model Wins",
        "W-L O/U",
        "Model - Vegas",
        "Actual - Vegas",
        "Actual Wins"
    ]
]

# Sort by actual standings
west_final = (
    west_final
    .sort_values("Actual Wins", ascending=False)
    .reset_index(drop=True)
)
# Clean numeric formatting
west_final["Model Wins"] = west_final["Model Wins"].astype(int)
west_final["Actual Wins"] = west_final["Actual Wins"].astype(int)
west_final["W-L O/U"] = west_final["W-L O/U"].astype(float)

west_final["Model - Vegas"] = west_final["Model - Vegas"].round(1)
west_final["Actual - Vegas"] = west_final["Actual - Vegas"].round(1)

# EAST

# Model standings
model_east = (
    pd.DataFrame(results[season]["standings"]["East"])
    .T
    .reset_index()
    .rename(columns={
        "index": "Team",
        "W_Pct": "W%"
    })
)

model_east = model_east[["Team", "W"]].rename(
    columns={"W": "Model Wins"}
)


# Actual standings
actual_east = (
    pd.DataFrame(nba_standings[season]["east"])
    .rename(columns={
        "teamAbbv": "Team",
        "W_Pct": "W%"
    })
)

actual_east = actual_east[["Team", "W"]].rename(
    columns={"W": "Actual Wins"}
)


# Vegas
vegas_east = (
    pd.DataFrame.from_dict(
        vegas_odds[season],
        orient="index"
    )
    .reset_index()
    .rename(columns={
        "index": "Team",
        "wl_ou": "W-L O/U"
    })
)

vegas_east = vegas_east[["Team", "W-L O/U"]]

# Merge
east_final = (
    model_east
    .merge(vegas_east, on="Team", how="left")
    .merge(actual_east, on="Team", how="left")
)

# Differences
east_final["Model - Vegas"] = (
    east_final["Model Wins"] - east_final["W-L O/U"]
)

east_final["Actual - Vegas"] = (
    east_final["Actual Wins"] - east_final["W-L O/U"]
)

# Final Column Order
east_final = east_final[
    [
        "Team",
        "Model Wins",
        "W-L O/U",
        "Model - Vegas",
        "Actual - Vegas",
        "Actual Wins"
    ]
]

# Sort by actual standings
east_final = (
    east_final
    .sort_values("Actual Wins", ascending=False)
    .reset_index(drop=True)
)

# Formatting
east_final["Model Wins"] = east_final["Model Wins"].astype(int)
east_final["Actual Wins"] = east_final["Actual Wins"].astype(int)
east_final["W-L O/U"] = east_final["W-L O/U"].astype(float)

east_final["Model - Vegas"] = east_final["Model - Vegas"].round(1)
east_final["Actual - Vegas"] = east_final["Actual - Vegas"].round(1)

In [ ]:
# Compare model vs vegas accuracies WEST 22_23
west_error = west_final.copy()

# Absolute error from actual win total
west_error["Model Abs Error"] = (
    west_error["Model Wins"] - west_error["Actual Wins"]
).abs()

west_error["Vegas Abs Error"] = (
    west_error["W-L O/U"] - west_error["Actual Wins"]
).abs()


# Summary statistics
west_stats = pd.DataFrame({
    "Mean": [
        west_error["Model Abs Error"].mean(),
        west_error["Vegas Abs Error"].mean()
    ],
    "Median": [
        west_error["Model Abs Error"].median(),
        west_error["Vegas Abs Error"].median()
    ],
    "Std Dev": [
        west_error["Model Abs Error"].std(),
        west_error["Vegas Abs Error"].std()
    ],
    "Min": [
        west_error["Model Abs Error"].min(),
        west_error["Vegas Abs Error"].min()
    ],
    "Max": [
        west_error["Model Abs Error"].max(),
        west_error["Vegas Abs Error"].max()
    ]
}, index=["Model", "Vegas"]).round(2)

display(west_final)
display(west_stats)

In [ ]:
# Compare model vs vegas accuracies East 22_23

east_error = east_final.copy()

# Absolute error from actual win total
east_error["Model Abs Error"] = (
    east_error["Model Wins"] - east_error["Actual Wins"]
).abs()

east_error["Vegas Abs Error"] = (
    east_error["W-L O/U"] - east_error["Actual Wins"]
).abs()


# Summary statistics
east_stats = pd.DataFrame({
    "Mean": [
        east_error["Model Abs Error"].mean(),
        east_error["Vegas Abs Error"].mean()
    ],
    "Median": [
        east_error["Model Abs Error"].median(),
        east_error["Vegas Abs Error"].median()
    ],
    "Std Dev": [
        east_error["Model Abs Error"].std(),
        east_error["Vegas Abs Error"].std()
    ],
    "Min": [
        east_error["Model Abs Error"].min(),
        east_error["Vegas Abs Error"].min()
    ],
    "Max": [
        east_error["Model Abs Error"].max(),
        east_error["Vegas Abs Error"].max()
    ]
}, index=["Model", "Vegas"]).round(2)

display(east_final)
display(east_stats)

In [239]:
# Go loop through the league schedule
# Gtab the two teams from the schedule
    # For each team, 
    # Get the 9 man roations (Helper fucntion)
    # Make sure the guys in the rotation subtract a game form their total 'G'
    # Get the cumulative team stats
    # Put both team stats into the model
    # LEAVEL MODEL PART BLANK FOR NOW
    # IF the model accurately presents i.e. one team wins, one team loses,
    # Go to standings and add accordingly using TeamAbbv
    # IF they dont align, pause and print
    # OTherwise so to league standings, update accordingly

# Get the two teams that are playing
# Get the 9 man roations for both teams 
# Subtract the GP by 1 for each player
# create a df called GameOutcomes
# gameoutcomes will store the model box score stats needed for the model to use to predict
# you will go into the scedule and for each 
# Therefore after everty game when a player reaches 0 games played, a new rotation needs to be made